In [ ]:
# ============================================================================
# 🧠 Subject Variability Analyzer - 受试者个体差异分析完整框架
# 
# 重新定位目标：
# 1. 量化受试者间个体差异对模型泛化的影响
# 2. 分析38号受试者的特殊性（异常值检测）
# 3. 评估Subject Embedding的必要性和潜力
# 4. 提供针对性的解决方案建议
# ============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import os
import time
import json
from datetime import datetime
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tqdm import tqdm
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score, 
                           confusion_matrix, classification_report, balanced_accuracy_score)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy

# 设置matplotlib中文显示
plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (15, 10)
plt.style.use('seaborn-v0_8')

class SubjectVariabilityAnalyzer:
    """
    受试者个体差异分析器
    
    核心功能：
    1. Leave-One-Subject-Out交叉验证分析
    2. 38号受试者异常性检测
    3. 受试者间相似性和聚类分析
    4. Subject Embedding必要性评估
    5. 年龄效应分析
    """
    
    def __init__(self, data_path, save_path='./subject_variability_analysis/'):
        self.data_path = data_path
        self.save_path = save_path
        self.setup_directories()
        
        # 分析结果存储
        self.results = {
            'loso_analysis': {},
            'subject_38_analysis': {},
            'subject_similarity_analysis': {},
            'age_effect_analysis': {},
            'embedding_necessity_analysis': {}
        }
        
        self.deep_config = {
                    'input_dim': 341,
                    'hidden_dim': 4096,
                    'num_classes': 102,
                    'dropout_rate': 0.5,
                    'learning_rate': 0.00001,
                    'weight_decay': 0.00001,
                    'batch_size': 128,
                    'epochs': 20,  # LOSO时可以用更少epoch
                    'early_stopping_patience': 5
        }
                
        # 设置设备
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"🔧 使用设备: {self.device}")
        
        print("🧠 受试者个体差异分析器初始化完成")
        print(f"📁 结果保存路径: {save_path}")

    class DeepRegModel(nn.Module):
        """你的4×4096深度学习架构"""
        def __init__(self, input_dim=341, hidden_dim=4096, num_classes=102, dropout_rate=0.5):
            super().__init__()
            self.fc1 = nn.Linear(input_dim, hidden_dim)
            self.fc2 = nn.Linear(hidden_dim, hidden_dim) 
            self.fc3 = nn.Linear(hidden_dim, hidden_dim)
            self.fc4 = nn.Linear(hidden_dim, hidden_dim)
            self.fc5 = nn.Linear(hidden_dim, num_classes)
            self.dropout = nn.Dropout(dropout_rate)
            
        def forward(self, x):
            x = self.dropout(F.relu(self.fc1(x)))
            x = self.dropout(F.relu(self.fc2(x)))
            x = self.dropout(F.relu(self.fc3(x)))
            x = self.dropout(F.relu(self.fc4(x)))
            x = self.fc5(x)
            return x

    def kernel_l2_regularization(self, model, weight_decay):
        """L2正则化（保持你的原始逻辑）"""
        l2_reg = 0
        for name, param in model.named_parameters():
            if 'weight' in name and param.requires_grad:
                l2_reg += torch.norm(param, p=2) ** 2
        return weight_decay * l2_reg


    def _train_and_evaluate_deep_model(self, X_train, y_train, X_test, y_test, 
                                    subject_id=None, verbose=False):
        """训练和评估单个深度学习模型"""
        
        # 创建模型
        model = self.DeepRegModel(
            input_dim=self.deep_config['input_dim'],
            hidden_dim=self.deep_config['hidden_dim'],
            num_classes=self.deep_config['num_classes'],
            dropout_rate=self.deep_config['dropout_rate']
        ).to(self.device)
        
        # 数据加载器
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train), 
            torch.LongTensor(y_train)
        )
        train_loader = DataLoader(
            train_dataset, 
            batch_size=self.deep_config['batch_size'], 
            shuffle=True
        )
        
        # 优化器和损失函数
        optimizer = optim.Adam(model.parameters(), lr=self.deep_config['learning_rate'])
        criterion = nn.CrossEntropyLoss()
        
        # 🔥 训练循环（简化版）
        model.train()
        for epoch in range(self.deep_config['epochs']):
            epoch_loss = 0
            
            for batch_data, batch_target in train_loader:
                batch_data = batch_data.to(self.device)
                batch_target = batch_target.to(self.device)
                
                optimizer.zero_grad()
                output = model(batch_data)
                
                # 计算损失（包含L2正则化）
                base_loss = criterion(output, batch_target)
                l2_reg = self.kernel_l2_regularization(model, self.deep_config['weight_decay'])
                total_loss = base_loss + l2_reg
                
                total_loss.backward()
                optimizer.step()
                
                epoch_loss += total_loss.item()
            
            if verbose and (epoch + 1) % 5 == 0:
                print(f"    Epoch {epoch+1}/{self.deep_config['epochs']}, Loss: {epoch_loss/len(train_loader):.4f}")
        
        # 🔥 评估
        model.eval()
        test_dataset = TensorDataset(
            torch.FloatTensor(X_test), 
            torch.LongTensor(y_test)
        )
        test_loader = DataLoader(test_dataset, batch_size=self.deep_config['batch_size'], shuffle=False)
        
        all_predictions = []
        all_targets = []
        
        with torch.no_grad():
            for batch_data, batch_target in test_loader:
                batch_data = batch_data.to(self.device)
                output = model(batch_data)
                predicted = torch.argmax(output, dim=1)
                
                all_predictions.extend(predicted.cpu().numpy())
                all_targets.extend(batch_target.numpy())
        
        # 计算指标
        accuracy = accuracy_score(all_targets, all_predictions)
        f1_macro = f1_score(all_targets, all_predictions, average='macro', zero_division=0)
        balanced_acc = balanced_accuracy_score(all_targets, all_predictions)
        
        # ✅ 修复：添加年龄信息
        subject_age = None
        if subject_id is not None and hasattr(self, 'subject_stats_df'):
            subject_info = self.subject_stats_df[self.subject_stats_df['subject_id'] == subject_id]
            if len(subject_info) > 0:
                subject_age = subject_info['age'].iloc[0]
        
        return {
            'subject_id': subject_id,
            'accuracy': accuracy,
            'f1_macro': f1_macro,
            'balanced_accuracy': balanced_acc,
            'n_test_samples': len(X_test),
            'age': subject_age  # ✅ 添加年龄字段
        }

        
    def setup_directories(self):
        """创建保存目录结构"""
        os.makedirs(self.save_path, exist_ok=True)
        os.makedirs(os.path.join(self.save_path, 'visualizations'), exist_ok=True)
        os.makedirs(os.path.join(self.save_path, 'models'), exist_ok=True)
        os.makedirs(os.path.join(self.save_path, 'reports'), exist_ok=True)
        
    def load_data_with_subjects_and_age(self):
        """
        加载数据并提取受试者ID和年龄信息
        """
        print("\n" + "="*80)
        print("📂 Phase 1: 数据加载与受试者信息提取")
        print("="*80)
        
        print("🔄 正在加载TRAIN38.mat数据...")
        
        try:
            with h5py.File(self.data_path, 'r') as f:
                # 加载主要数据
                data = np.array(f['data']).transpose()
                region = np.array(f['region']).transpose()
                prob_idx = np.array(f['prob_idx']).transpose().flatten().astype(int)
                all_age = np.array(f['all_age']).transpose().flatten()
                
                print(f"✅ 数据加载成功")
                print(f"  - 数据矩阵: {data.shape}")
                print(f"  - 标签矩阵: {region.shape}")
                print(f"  - 受试者ID: {prob_idx.shape}, 范围: [{prob_idx.min()}, {prob_idx.max()}]")
                print(f"  - 年龄信息: {all_age.shape}, 范围: [{all_age.min():.1f}, {all_age.max():.1f}]")
                
        except Exception as e:
            print(f"❌ 数据加载失败: {e}")
            raise
        
        # 数据预处理
        print("\n🔧 数据预处理...")
        
        # 处理标签
        if len(region.shape) > 1 and region.shape[1] > 1:
            # one-hot转换为类别索引
            y_labels = np.argmax(region, axis=1)
        else:
            y_labels = region.flatten().astype(int)
        
        # 标准化特征
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(data)
        
        # 存储数据
        self.data = {
            'X': X_scaled,
            'y': y_labels,
            'y_onehot': region,
            'subject_ids': prob_idx,
            'ages': all_age,
            'scaler': scaler,
            'n_subjects': len(np.unique(prob_idx)),
            'n_classes': len(np.unique(y_labels)),
            'n_samples': len(data),
            'n_features': data.shape[1]
        }
        
        print(f"✅ 数据预处理完成")
        print(f"  - 受试者数量: {self.data['n_subjects']}")
        print(f"  - 类别数量: {self.data['n_classes']}")
        print(f"  - 总样本数: {self.data['n_samples']:,}")
        print(f"  - 特征维度: {self.data['n_features']}")
        
        # 受试者信息统计
        self._generate_subject_statistics()
        
        return self.data
    
    def _generate_subject_statistics(self):
        """生成受试者统计信息"""
        print("\n📊 受试者信息统计:")
        subject_stats = []
        unique_subjects = np.unique(self.data['subject_ids'])
        
        for subject_id in unique_subjects:
            mask = self.data['subject_ids'] == subject_id
            n_samples = np.sum(mask)
            age = self.data['ages'][mask][0]
            
            # 计算该受试者的类别分布
            subject_labels = self.data['y'][mask]
            class_distribution = np.bincount(subject_labels, minlength=self.data['n_classes'])
            
            subject_stats.append({
                'subject_id': subject_id,
                'n_samples': n_samples,
                'age': age,
                'percentage': n_samples / len(self.data['X']) * 100,
                'class_distribution': class_distribution,
                'n_classes_present': np.sum(class_distribution > 0)
            })
        
        self.subject_stats_df = pd.DataFrame(subject_stats)
        
        print(f"  - 平均每受试者样本数: {self.subject_stats_df['n_samples'].mean():.0f}")
        print(f"  - 样本数范围: [{self.subject_stats_df['n_samples'].min():,}, {self.subject_stats_df['n_samples'].max():,}]")
        print(f"  - 年龄范围: [{self.subject_stats_df['age'].min():.1f}, {self.subject_stats_df['age'].max():.1f}] 岁")
        print(f"  - 平均年龄: {self.subject_stats_df['age'].mean():.1f} ± {self.subject_stats_df['age'].std():.1f} 岁")
        
        # 特别关注38号受试者
        subject_38_info = self.subject_stats_df[self.subject_stats_df['subject_id'] == 38]
        if len(subject_38_info) > 0:
            print(f"  - 38号受试者: {subject_38_info['n_samples'].iloc[0]:,} 样本, 年龄 {subject_38_info['age'].iloc[0]:.1f} 岁")
            age_diff_from_mean = subject_38_info['age'].iloc[0] - self.subject_stats_df[self.subject_stats_df['subject_id'] != 38]['age'].mean()
            print(f"  - 38号年龄差异: {age_diff_from_mean:+.1f} 岁 (相对于1-37号平均)")

    def leave_one_subject_out_analysis(self):
        """
        🎯 深度学习版本的Leave-One-Subject-Out交叉验证分析
        """
        print("\n" + "="*80)
        print("🔍 Phase 2: Deep Learning LOSO 交叉验证分析")
        print("="*80)
        
        print("🔄 开始深度学习LOSO分析...")
        print("📊 这将告诉我们：")
        print("  1. 你的深度架构在严格受试者分离下的真实性能")
        print("  2. 受试者38是否异常")
        print("  3. 是否需要Subject Embedding")
        
        # 获取1-37号受试者（训练集受试者）
        training_subjects = [s for s in np.unique(self.data['subject_ids']) if s != 38]
        
        print(f"📊 将分析 {len(training_subjects)} 个训练受试者")
        
        loso_results = []
        
        # === 核心LOSO循环 ===
        for i, test_subject in enumerate(training_subjects):
            print(f"\n📊 LOSO {i+1}/{len(training_subjects)}: 测试受试者 {test_subject}")
            
            # 分割数据
            train_mask = self.data['subject_ids'] != test_subject
            test_mask = self.data['subject_ids'] == test_subject
            
            X_train = self.data['X'][train_mask]
            y_train = self.data['y'][train_mask]
            X_test = self.data['X'][test_mask]
            y_test = self.data['y'][test_mask]
            
            print(f"  训练样本: {len(X_train):,}, 测试样本: {len(X_test):,}")
            
            try:
                # 🔥 训练你的深度学习模型
                model_results = self._train_and_evaluate_deep_model(
                    X_train, y_train, X_test, y_test, 
                    subject_id=test_subject,
                    verbose=(i < 3)  # 前3个显示详细进度
                )
                
                loso_results.append(model_results)
                
                print(f"  ✅ F1: {model_results['f1_macro']:.4f}, "
                      f"Acc: {model_results['accuracy']:.4f}")
                
            except Exception as e:
                print(f"  ❌ 失败: {e}")
                continue
        
        # === 单独测试38号受试者 ===
        print(f"\n🎯 特别测试: 受试者38")
        
        train_mask_38 = self.data['subject_ids'] != 38
        test_mask_38 = self.data['subject_ids'] == 38
        
        X_train_38 = self.data['X'][train_mask_38]
        y_train_38 = self.data['y'][train_mask_38]
        X_test_38 = self.data['X'][test_mask_38]
        y_test_38 = self.data['y'][test_mask_38]
        
        try:
            subject_38_result = self._train_and_evaluate_deep_model(
                X_train_38, y_train_38, X_test_38, y_test_38,
                subject_id=38, verbose=True
            )
            print(f"  ✅ 受试者38 F1: {subject_38_result['f1_macro']:.4f}")
        except Exception as e:
            print(f"  ❌ 受试者38失败: {e}")
            subject_38_result = None
        
        # === 计算统计结果 ===
        if loso_results:
            f1_scores = [r['f1_macro'] for r in loso_results]
            
            loso_stats = {
                'mean_f1_macro': np.mean(f1_scores),
                'std_f1_macro': np.std(f1_scores),
                'min_f1_macro': np.min(f1_scores),
                'max_f1_macro': np.max(f1_scores),
                'individual_results': loso_results,
                'subject_38_result': subject_38_result,
                'n_subjects_tested': len(loso_results)
            }
            
            print(f"\n📊 深度学习LOSO统计:")
            print(f"  平均F1: {loso_stats['mean_f1_macro']:.4f} ± {loso_stats['std_f1_macro']:.4f}")
            print(f"  F1范围: [{loso_stats['min_f1_macro']:.4f}, {loso_stats['max_f1_macro']:.4f}]")
            
            if subject_38_result:
                gap = loso_stats['mean_f1_macro'] - subject_38_result['f1_macro']
                gap_in_std = gap / loso_stats['std_f1_macro'] if loso_stats['std_f1_macro'] > 0 else 0
                
                print(f"  受试者38 F1: {subject_38_result['f1_macro']:.4f}")
                print(f"  38号gap: {gap:+.4f} ({gap_in_std:+.1f}σ)")
                
                # 🎯 核心判断
                if gap_in_std > 2.0:
                    print(f"  🚨 受试者38显著异常 (>2σ)")
                elif gap_in_std > 1.5:
                    print(f"  ⚠️ 受试者38可能异常 (>1.5σ)")
                else:
                    print(f"  ✅ 受试者38在正常范围内")
            
            self.results['loso_analysis'] = {'DeepLearning': loso_stats}
            return loso_stats
        
        else:
            print("❌ 没有成功的LOSO结果")
            return None
        
        
    def _generate_loso_summary(self):
        """生成LOSO分析总结"""
        print(f"\n📊 LOSO分析总结:")
        print("="*50)
        
        loso_results = self.results['loso_analysis']
        
        for model_name, results in loso_results.items():
            if model_name == 'DeepLearning' and results['subject_38_result']:  # ✅ 明确指定
                loso_mean = results['mean_f1_macro']
                subject_38_f1 = results['subject_38_result']['f1_macro']
                gap = loso_mean - subject_38_f1
                gap_in_std = gap / results['std_f1_macro'] if results['std_f1_macro'] > 0 else 0
                
                print(f"\n{model_name}:")
                print(f"  LOSO平均F1: {loso_mean:.4f}")
                print(f"  38号F1: {subject_38_f1:.4f}")
                print(f"  性能gap: {gap:.4f} ({gap_in_std:.1f} 个标准差)")
                
                # 判断38号是否异常
                if gap_in_std > 2.0:
                    print(f"  🚨 38号显著异常 (>2σ)")
                elif gap_in_std > 1.5:
                    print(f"  ⚠️ 38号可能异常 (>1.5σ)")
                elif gap < results['std_f1_macro']:
                    print(f"  ✅ 38号在正常范围内")
                else:
                    print(f"  📊 38号表现较差但不算异常")
        
        # ✅ 修复：关键洞察部分
        dl_results = loso_results.get('DeepLearning')  # ✅ 改成DeepLearning
        if dl_results and dl_results['subject_38_result']:
            loso_mean = dl_results['mean_f1_macro']
            subject_38_f1 = dl_results['subject_38_result']['f1_macro']
            
            print(f"\n🎯 关键洞察:")
            if loso_mean < 0.60:
                print(f"  📊 LOSO平均性能较低 ({loso_mean:.3f})，受试者差异是普遍问题")
                print(f"  💡 强烈建议Subject Embedding")
            elif loso_mean > 0.70:
                print(f"  📊 LOSO平均性能良好 ({loso_mean:.3f})，38号可能是特殊情况")
                print(f"  💡 重点分析38号数据质量")
            else:
                print(f"  📊 LOSO平均性能中等 ({loso_mean:.3f})，需要综合分析")
            
            if abs(subject_38_f1 - loso_mean) > 0.10:
                print(f"  🎯 38号与LOSO差异显著，Subject Embedding很有必要")
            else:
                print(f"  📋 38号差异在可接受范围，可能需要其他方法")

    def analyze_subject_38_anomaly(self):
        """
        深度分析38号受试者的异常性
        """
        print("\n" + "="*80)
        print("🔍 Phase 3: 38号受试者异常性分析")
        print("="*80)
        
        # 获取38号受试者数据
        subject_38_mask = self.data['subject_ids'] == 38
        training_subjects_mask = self.data['subject_ids'] != 38
        
        X_38 = self.data['X'][subject_38_mask]
        X_training = self.data['X'][training_subjects_mask]
        
        analysis_results = {}
        
        # 1. 年龄差异分析
        print("\n📊 1. 年龄差异分析")
        subject_38_age = self.subject_stats_df[self.subject_stats_df['subject_id'] == 38]['age'].iloc[0]
        training_ages = self.subject_stats_df[self.subject_stats_df['subject_id'] != 38]['age'].values
        
        age_analysis = {
            'subject_38_age': subject_38_age,
            'training_mean_age': np.mean(training_ages),
            'training_std_age': np.std(training_ages),
            'age_difference': subject_38_age - np.mean(training_ages),
            'age_z_score': (subject_38_age - np.mean(training_ages)) / np.std(training_ages),
            'training_age_range': [np.min(training_ages), np.max(training_ages)]
        }
        
        print(f"  38号年龄: {subject_38_age:.1f} 岁")
        print(f"  训练集平均年龄: {age_analysis['training_mean_age']:.1f} ± {age_analysis['training_std_age']:.1f} 岁")
        print(f"  年龄差异: {age_analysis['age_difference']:+.1f} 岁")
        print(f"  年龄Z分数: {age_analysis['age_z_score']:+.2f}")
        
        if abs(age_analysis['age_z_score']) > 2:
            print(f"  🚨 年龄显著异常 (|Z| > 2)")
        elif abs(age_analysis['age_z_score']) > 1.5:
            print(f"  ⚠️ 年龄可能异常 (|Z| > 1.5)")
        else:
            print(f"  ✅ 年龄在正常范围内")
        
        analysis_results['age_analysis'] = age_analysis
        
        # 2. 特征分布异常检测
        print("\n📊 2. 特征分布异常检测")
        
        # 计算38号相对于训练集的特征异常程度
        training_mean = np.mean(X_training, axis=0)
        training_std = np.std(X_training, axis=0)
        subject_38_mean = np.mean(X_38, axis=0)
        
        # 特征级别的Z分数
        feature_z_scores = (subject_38_mean - training_mean) / (training_std + 1e-8)
        
        # 异常特征统计
        outlier_features_2sigma = np.sum(np.abs(feature_z_scores) > 2)
        outlier_features_3sigma = np.sum(np.abs(feature_z_scores) > 3)
        
        feature_analysis = {
            'feature_z_scores': feature_z_scores,
            'outlier_features_2sigma': outlier_features_2sigma,
            'outlier_features_3sigma': outlier_features_3sigma,
            'outlier_percentage_2sigma': outlier_features_2sigma / len(feature_z_scores) * 100,
            'outlier_percentage_3sigma': outlier_features_3sigma / len(feature_z_scores) * 100,
            'max_z_score': np.max(np.abs(feature_z_scores)),
            'most_outlier_features': np.argsort(np.abs(feature_z_scores))[-10:]  # 最异常的10个特征
        }
        
        print(f"  异常特征 (|Z| > 2): {outlier_features_2sigma}/{len(feature_z_scores)} ({feature_analysis['outlier_percentage_2sigma']:.1f}%)")
        print(f"  严重异常特征 (|Z| > 3): {outlier_features_3sigma}/{len(feature_z_scores)} ({feature_analysis['outlier_percentage_3sigma']:.1f}%)")
        print(f"  最大|Z|分数: {feature_analysis['max_z_score']:.2f}")
        
        if feature_analysis['outlier_percentage_2sigma'] > 20:
            print(f"  🚨 特征分布显著异常 (>20%特征异常)")
        elif feature_analysis['outlier_percentage_2sigma'] > 10:
            print(f"  ⚠️ 特征分布可能异常 (>10%特征异常)")
        else:
            print(f"  ✅ 特征分布相对正常")
        
        analysis_results['feature_analysis'] = feature_analysis
        
        # 3. 样本级别异常检测
        print("\n📊 3. 样本级别异常检测")
        
        # 计算38号每个样本到训练集的距离
        from sklearn.neighbors import NearestNeighbors
        
        # 使用KNN计算异常分数
        nn = NearestNeighbors(n_neighbors=min(100, len(X_training)//10))
        nn.fit(X_training)
        
        distances_38, _ = nn.kneighbors(X_38)
        mean_distances_38 = np.mean(distances_38, axis=1)
        
        # 训练集内部的距离作为基准
        distances_training, _ = nn.kneighbors(X_training)
        mean_distances_training = np.mean(distances_training, axis=1)
        
        sample_analysis = {
            'mean_distance_38': np.mean(mean_distances_38),
            'std_distance_38': np.std(mean_distances_38),
            'mean_distance_training': np.mean(mean_distances_training),
            'std_distance_training': np.std(mean_distances_training),
            'distance_z_score': (np.mean(mean_distances_38) - np.mean(mean_distances_training)) / np.std(mean_distances_training),
            'outlier_samples_percentage': np.sum(mean_distances_38 > np.mean(mean_distances_training) + 2*np.std(mean_distances_training)) / len(mean_distances_38) * 100
        }
        
        print(f"  38号平均距离: {sample_analysis['mean_distance_38']:.4f}")
        print(f"  训练集平均距离: {sample_analysis['mean_distance_training']:.4f}")
        print(f"  距离Z分数: {sample_analysis['distance_z_score']:+.2f}")
        print(f"  异常样本比例: {sample_analysis['outlier_samples_percentage']:.1f}%")
        
        if sample_analysis['distance_z_score'] > 2:
            print(f"  🚨 38号样本显著远离训练分布")
        elif sample_analysis['distance_z_score'] > 1:
            print(f"  ⚠️ 38号样本可能偏离训练分布")
        else:
            print(f"  ✅ 38号样本在正常范围内")
        
        analysis_results['sample_analysis'] = sample_analysis
        
        # 4. 类别分布比较
        print("\n📊 4. 类别分布比较")
        
        subject_38_classes = self.data['y'][subject_38_mask]
        training_classes = self.data['y'][training_subjects_mask]
        
        # 计算类别分布
        subject_38_class_dist = np.bincount(subject_38_classes, minlength=self.data['n_classes'])
        training_class_dist = np.bincount(training_classes, minlength=self.data['n_classes'])
        
        # 归一化为概率分布
        subject_38_class_prob = subject_38_class_dist / np.sum(subject_38_class_dist)
        training_class_prob = training_class_dist / np.sum(training_class_dist)
        
        # 计算KL散度
        from scipy.stats import entropy
        kl_divergence = entropy(subject_38_class_prob + 1e-10, training_class_prob + 1e-10)
        
        class_analysis = {
            'subject_38_class_dist': subject_38_class_dist,
            'training_class_dist': training_class_dist,
            'subject_38_class_prob': subject_38_class_prob,
            'training_class_prob': training_class_prob,
            'kl_divergence': kl_divergence,
            'classes_present_38': np.sum(subject_38_class_dist > 0),
            'classes_present_training': np.sum(training_class_dist > 0),
            'unique_to_38': np.sum((subject_38_class_dist > 0) & (training_class_dist == 0)),
            'missing_in_38': np.sum((subject_38_class_dist == 0) & (training_class_dist > 0))
        }
        
        print(f"  38号类别数: {class_analysis['classes_present_38']}")
        print(f"  训练集类别数: {class_analysis['classes_present_training']}")
        print(f"  KL散度: {kl_divergence:.4f}")
        print(f"  38号独有类别: {class_analysis['unique_to_38']}")
        print(f"  38号缺失类别: {class_analysis['missing_in_38']}")
        
        if kl_divergence > 1.0:
            print(f"  🚨 类别分布显著不同")
        elif kl_divergence > 0.5:
            print(f"  ⚠️ 类别分布有一定差异")
        else:
            print(f"  ✅ 类别分布相对相似")
        
        analysis_results['class_analysis'] = class_analysis
        
        # 5. 综合异常性评分
        print("\n📊 5. 综合异常性评分")
        
        anomaly_score = 0
        anomaly_reasons = []
        
        # 年龄异常贡献
        if abs(age_analysis['age_z_score']) > 2:
            anomaly_score += 3
            anomaly_reasons.append("年龄显著异常")
        elif abs(age_analysis['age_z_score']) > 1.5:
            anomaly_score += 2
            anomaly_reasons.append("年龄可能异常")
        
        # 特征异常贡献
        if feature_analysis['outlier_percentage_2sigma'] > 20:
            anomaly_score += 3
            anomaly_reasons.append("特征分布显著异常")
        elif feature_analysis['outlier_percentage_2sigma'] > 10:
            anomaly_score += 2
            anomaly_reasons.append("特征分布可能异常")
        
        # 样本距离异常贡献
        if sample_analysis['distance_z_score'] > 2:
            anomaly_score += 3
            anomaly_reasons.append("样本距离显著异常")
        elif sample_analysis['distance_z_score'] > 1:
            anomaly_score += 2
            anomaly_reasons.append("样本距离可能异常")
        
        # 类别分布异常贡献
        if kl_divergence > 1.0:
            anomaly_score += 2
            anomaly_reasons.append("类别分布显著不同")
        
        anomaly_analysis = {
            'anomaly_score': anomaly_score,
            'anomaly_reasons': anomaly_reasons,
            'max_possible_score': 11
        }
        
        print(f"  综合异常分数: {anomaly_score}/11")
        print(f"  异常原因: {', '.join(anomaly_reasons) if anomaly_reasons else '无显著异常'}")
        
        if anomaly_score >= 8:
            print(f"  🚨 38号高度异常，建议检查数据质量")
        elif anomaly_score >= 5:
            print(f"  ⚠️ 38号中度异常，需要特殊处理")
        elif anomaly_score >= 3:
            print(f"  📊 38号轻度异常，在可接受范围")
        else:
            print(f"  ✅ 38号基本正常")
        
        analysis_results['anomaly_analysis'] = anomaly_analysis
        
        self.results['subject_38_analysis'] = analysis_results
        return analysis_results
    
    def analyze_subject_similarity_and_clustering(self):
        """
        受试者相似性和聚类分析
        """
        print("\n" + "="*80)
        print("🔍 Phase 4: 受试者相似性和聚类分析")
        print("="*80)
        
        # 计算每个受试者的平均特征向量
        unique_subjects = np.unique(self.data['subject_ids'])
        subject_profiles = {}
        subject_ages = {}
        
        print("🔄 计算受试者特征档案...")
        for subject_id in unique_subjects:
            mask = self.data['subject_ids'] == subject_id
            if np.sum(mask) > 10:  # 确保有足够样本
                subject_profiles[subject_id] = np.mean(self.data['X'][mask], axis=0)
                subject_ages[subject_id] = self.data['ages'][mask][0]
        
        # 构建特征矩阵
        profile_subjects = list(subject_profiles.keys())
        profile_matrix = np.array([subject_profiles[sid] for sid in profile_subjects])
        ages_array = np.array([subject_ages[sid] for sid in profile_subjects])
        
        print(f"  成功构建 {len(profile_subjects)} 个受试者的特征档案")
        
        similarity_results = {}
        
        # 1. 相似性矩阵计算
        print("\n📊 1. 计算受试者间相似性...")
        
        # 欧氏距离
        from scipy.spatial.distance import pdist, squareform
        distance_matrix = squareform(pdist(profile_matrix, metric='euclidean'))
        
        # 余弦相似性
        from sklearn.metrics.pairwise import cosine_similarity
        cosine_sim_matrix = cosine_similarity(profile_matrix)
        
        # 皮尔逊相关性
        correlation_matrix = np.corrcoef(profile_matrix)
        
        similarity_results['distance_matrix'] = distance_matrix
        similarity_results['cosine_similarity'] = cosine_sim_matrix
        similarity_results['correlation_matrix'] = correlation_matrix
        similarity_results['subject_ids'] = profile_subjects
        
        # 2. 聚类分析
        print("\n📊 2. 受试者聚类分析...")
        
        # 层次聚类
        from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
        linkage_matrix = linkage(profile_matrix, method='ward')
        
        # 尝试不同的聚类数
        cluster_results = {}
        silhouette_scores = []
        
        for n_clusters in range(2, min(8, len(profile_subjects)//2)):
            cluster_labels = fcluster(linkage_matrix, n_clusters, criterion='maxclust')
            
            from sklearn.metrics import silhouette_score
            sil_score = silhouette_score(profile_matrix, cluster_labels)
            silhouette_scores.append(sil_score)
            
            cluster_results[n_clusters] = {
                'labels': cluster_labels,
                'silhouette_score': sil_score
            }
        
        best_n_clusters = range(2, min(8, len(profile_subjects)//2))[np.argmax(silhouette_scores)]
        best_silhouette = max(silhouette_scores)
        
        print(f"  最优聚类数: {best_n_clusters} (轮廓系数: {best_silhouette:.3f})")
        
        similarity_results['clustering'] = {
            'linkage_matrix': linkage_matrix,
            'best_n_clusters': best_n_clusters,
            'best_silhouette': best_silhouette,
            'cluster_results': cluster_results,
            'silhouette_scores': silhouette_scores
        }
        
        # 3. 38号受试者的相似性分析
        print("\n📊 3. 38号受试者相似性分析...")
        
        if 38 in profile_subjects:
            subject_38_idx = profile_subjects.index(38)
            
            # 找到与38号最相似的受试者
            distances_to_38 = distance_matrix[subject_38_idx]
            cosine_sim_to_38 = cosine_sim_matrix[subject_38_idx]
            correlation_to_38 = correlation_matrix[subject_38_idx]
            
            # 排除38号自己
            other_subjects_mask = np.arange(len(profile_subjects)) != subject_38_idx
            
            most_similar_idx = np.argmin(distances_to_38[other_subjects_mask])
            most_similar_subject = [sid for i, sid in enumerate(profile_subjects) if i != subject_38_idx][most_similar_idx]
            
            least_similar_idx = np.argmax(distances_to_38[other_subjects_mask])
            least_similar_subject = [sid for i, sid in enumerate(profile_subjects) if i != subject_38_idx][least_similar_idx]
            
            similarity_38_results = {
                'most_similar_subject': most_similar_subject,
                'least_similar_subject': least_similar_subject,
                'min_distance': np.min(distances_to_38[other_subjects_mask]),
                'max_distance': np.max(distances_to_38[other_subjects_mask]),
                'mean_distance': np.mean(distances_to_38[other_subjects_mask]),
                'distance_rank': np.sum(distances_to_38[other_subjects_mask] < distances_to_38[subject_38_idx]),
                'max_cosine_similarity': np.max(cosine_sim_to_38[other_subjects_mask]),
                'max_correlation': np.max(correlation_to_38[other_subjects_mask])
            }
            
            print(f"  最相似受试者: {most_similar_subject} (距离: {similarity_38_results['min_distance']:.4f})")
            print(f"  最不相似受试者: {least_similar_subject} (距离: {similarity_38_results['max_distance']:.4f})")
            print(f"  平均距离: {similarity_38_results['mean_distance']:.4f}")
            print(f"  38号在距离排序中的位置: {similarity_38_results['distance_rank']}/{len(profile_subjects)-1}")
            
            similarity_results['subject_38_similarity'] = similarity_38_results
        
        # 4. 年龄与相似性的关系
        print("\n📊 4. 年龄与相似性关系分析...")
        
        age_similarity_corr = []
        for i in range(len(profile_subjects)):
            for j in range(i+1, len(profile_subjects)):
                age_diff = abs(ages_array[i] - ages_array[j])
                feature_distance = distance_matrix[i, j]
                age_similarity_corr.append((age_diff, feature_distance))
        
        age_diffs, feature_distances = zip(*age_similarity_corr)
        age_distance_correlation = np.corrcoef(age_diffs, feature_distances)[0, 1]
        
        print(f"  年龄差异与特征距离相关性: {age_distance_correlation:.4f}")
        
        if age_distance_correlation > 0.3:
            print(f"  📊 年龄对受试者差异有显著影响")
        elif age_distance_correlation > 0.1:
            print(f"  📊 年龄对受试者差异有一定影响")
        else:
            print(f"  📊 年龄对受试者差异影响较小")
        
        similarity_results['age_distance_correlation'] = age_distance_correlation
        similarity_results['ages'] = ages_array
        
        self.results['subject_similarity_analysis'] = similarity_results
        return similarity_results
    
    def analyze_age_effects(self):
        """
        年龄效应专项分析
        """
        print("\n" + "="*80)
        print("🔍 Phase 5: 年龄效应专项分析")
        print("="*80)
        
        age_results = {}
        
        # 1. 年龄分布分析
        print("📊 1. 年龄分布分析")
        
        subject_38_age = self.subject_stats_df[self.subject_stats_df['subject_id'] == 38]['age'].iloc[0]
        training_ages = self.subject_stats_df[self.subject_stats_df['subject_id'] != 38]['age'].values
        
        age_distribution_analysis = {
            'subject_38_age': subject_38_age,
            'training_ages': training_ages,
            'training_mean': np.mean(training_ages),
            'training_std': np.std(training_ages),
            'training_range': [np.min(training_ages), np.max(training_ages)],
            'age_gap': subject_38_age - np.mean(training_ages),
            'age_z_score': (subject_38_age - np.mean(training_ages)) / np.std(training_ages)
        }
        
        print(f"  训练集年龄: {age_distribution_analysis['training_mean']:.1f} ± {age_distribution_analysis['training_std']:.1f} 岁")
        print(f"  训练集范围: [{age_distribution_analysis['training_range'][0]:.1f}, {age_distribution_analysis['training_range'][1]:.1f}] 岁")
        print(f"  38号年龄: {subject_38_age:.1f} 岁")
        print(f"  年龄差距: {age_distribution_analysis['age_gap']:+.1f} 岁")
        
        age_results['age_distribution'] = age_distribution_analysis
        
        # 2. 基于年龄的性能预期模型
        print("\n📊 2. 基于年龄的性能预期分析")
        
        # 如果有LOSO结果,分析年龄与性能的关系
        if 'loso_analysis' in self.results and 'DeepLearning' in self.results['loso_analysis']:    # ✅ 改成这行
            loso_results = self.results['loso_analysis']['DeepLearning']  # ✅ 改成这行
            individual_results = loso_results['individual_results']
            # 提取年龄和性能数据
            ages = [r['age'] for r in individual_results]
            f1_scores = [r['f1_macro'] for r in individual_results]
            
            # 计算年龄与性能的相关性
            age_performance_corr = np.corrcoef(ages, f1_scores)[0, 1]
            
            # 线性回归预测38号性能
            from sklearn.linear_model import LinearRegression
            lr = LinearRegression()
            lr.fit(np.array(ages).reshape(-1, 1), f1_scores)
            
            predicted_38_performance = lr.predict([[subject_38_age]])[0]
            actual_38_performance = loso_results['subject_38_result']['f1_macro'] if loso_results['subject_38_result'] else None
            
            age_performance_analysis = {
                'age_performance_correlation': age_performance_corr,
                'predicted_38_performance': predicted_38_performance,
                'actual_38_performance': actual_38_performance,
                'age_slope': lr.coef_[0],
                'age_intercept': lr.intercept_
            }
            
            print(f"  年龄-性能相关性: {age_performance_corr:.4f}")
            print(f"  基于年龄预测38号性能: {predicted_38_performance:.4f}")
            if actual_38_performance:
                residual = actual_38_performance - predicted_38_performance
                print(f"  实际38号性能: {actual_38_performance:.4f}")
                print(f"  年龄解释后的残差: {residual:+.4f}")
                
                if abs(residual) < 0.03:
                    print("  ✅ 年龄基本解释了38号的性能差异")
                elif abs(residual) < 0.05:
                    print("  📊 年龄部分解释了38号的性能差异")
                else:
                    print("  ⚠️ 年龄无法充分解释38号的性能差异")
            
            age_results['age_performance'] = age_performance_analysis
        
        # 3. 年龄校正建议
        print("\n📊 3. 年龄校正建议")
        
        age_gap = abs(age_distribution_analysis['age_gap'])
        
        if age_gap > 10:
            correction_priority = "高"
            correction_methods = ["年龄回归校正", "年龄分层建模", "年龄权重调整"]
        elif age_gap > 5:
            correction_priority = "中"
            correction_methods = ["年龄协变量", "年龄正规化"]
        else:
            correction_priority = "低"
            correction_methods = ["可选择性年龄校正"]
        
        age_correction_recommendations = {
            'correction_priority': correction_priority,
            'recommended_methods': correction_methods,
            'expected_improvement': self._estimate_age_correction_benefit(age_gap)
        }
        
        print(f"  年龄校正优先级: {correction_priority}")
        print(f"  推荐方法: {', '.join(correction_methods)}")
        print(f"  预期改善幅度: {age_correction_recommendations['expected_improvement']:.3f}")
        
        age_results['correction_recommendations'] = age_correction_recommendations
        
        self.results['age_effect_analysis'] = age_results
        return age_results 
    
    def _estimate_age_correction_benefit(self, age_gap):
        """估算年龄校正的潜在收益"""
        # 基于研究文献的经验公式
        if age_gap > 10:
            return min(0.10, age_gap * 0.008)  # 最多10%改善
        elif age_gap > 5:
            return min(0.05, age_gap * 0.006)  # 最多5%改善
        else:
            return age_gap * 0.003  # 小幅改善

    def evaluate_embedding_necessity(self):
        """
        评估Subject Embedding的必要性和设计建议
        """
        print("\n" + "="*80)
        print("🔍 Phase 6: Subject Embedding必要性评估")
        print("="*80)
        
        embedding_results = {}
        
        # 1. 基于LOSO结果的必要性评估
        print("📊 1. 基于LOSO分析的必要性评估")
        
        if 'loso_analysis' in self.results:
            dl_results = self.results['loso_analysis'].get('DeepLearning')    # ✅ 改成这行
            if dl_results and dl_results['subject_38_result']:
                loso_mean = dl_results['mean_f1_macro']
                loso_std = dl_results['std_f1_macro']
                subject_38_f1 = dl_results['subject_38_result']['f1_macro']

                
                # 计算受试者变异性
                subject_variability = loso_std / loso_mean if loso_mean > 0 else 0
                
                # 计算38号偏离程度
                deviation_38 = (loso_mean - subject_38_f1) / loso_std if loso_std > 0 else 0
                
                necessity_score = 0
                necessity_reasons = []
                
                if subject_variability > 0.2:
                    necessity_score += 3
                    necessity_reasons.append("受试者间变异性高")
                elif subject_variability > 0.1:
                    necessity_score += 2
                    necessity_reasons.append("受试者间变异性中等")
                
                if deviation_38 > 2:
                    necessity_score += 3
                    necessity_reasons.append("目标受试者显著偏离")
                elif deviation_38 > 1:
                    necessity_score += 2
                    necessity_reasons.append("目标受试者适度偏离")
                
                if loso_mean < 0.6:
                    necessity_score += 2
                    necessity_reasons.append("整体跨受试者性能较低")
                
                loso_necessity = {
                    'necessity_score': necessity_score,
                    'max_score': 8,
                    'necessity_reasons': necessity_reasons,
                    'subject_variability': subject_variability,
                    'deviation_38': deviation_38
                }
                
                print(f"  受试者变异性: {subject_variability:.3f}")
                print(f"  38号偏离程度: {deviation_38:.1f} 标准差")
                print(f"  必要性得分: {necessity_score}/8")
                print(f"  必要性原因: {', '.join(necessity_reasons)}")
                
                embedding_results['loso_necessity'] = loso_necessity
        
        # 2. 基于受试者相似性分析的评估
        print("\n📊 2. 基于受试者相似性的评估")
        
        if 'subject_similarity_analysis' in self.results:
            similarity_results = self.results['subject_similarity_analysis']
            
            # 聚类质量
            clustering = similarity_results['clustering']
            best_silhouette = clustering['best_silhouette']
            
            # 38号的相似性
            if 'subject_38_similarity' in similarity_results:
                subject_38_sim = similarity_results['subject_38_similarity']
                distance_rank = subject_38_sim['distance_rank']
                total_subjects = len(similarity_results['subject_ids']) - 1
                distance_percentile = distance_rank / total_subjects
                
                similarity_necessity = {
                    'clustering_quality': best_silhouette,
                    'distance_percentile_38': distance_percentile,
                    'embedding_architecture_suggestion': self._suggest_embedding_architecture(best_silhouette, distance_percentile)
                }
                
                print(f"  聚类质量: {best_silhouette:.3f}")
                print(f"  38号距离百分位: {distance_percentile:.1%}")
                
                embedding_results['similarity_necessity'] = similarity_necessity
        
        # 3. 综合必要性评估
        print("\n📊 3. 综合必要性评估和建议")
        
        total_necessity_score = 0
        max_total_score = 0
        
        if 'loso_necessity' in embedding_results:
            total_necessity_score += embedding_results['loso_necessity']['necessity_score']
            max_total_score += embedding_results['loso_necessity']['max_score']
        
        # 年龄效应贡献
        if 'age_effect_analysis' in self.results:
            age_analysis = self.results['age_effect_analysis']['age_distribution']
            if abs(age_analysis['age_z_score']) > 1.5:
                total_necessity_score += 2
                max_total_score += 2
        
        # 异常性贡献
        if 'subject_38_analysis' in self.results:
            anomaly_analysis = self.results['subject_38_analysis']['anomaly_analysis']
            if anomaly_analysis['anomaly_score'] > 5:
                total_necessity_score += 2
                max_total_score += 2
        
        necessity_percentage = total_necessity_score / max_total_score * 100 if max_total_score > 0 else 0
        
        # 生成建议
        recommendations = self._generate_embedding_recommendations(necessity_percentage, embedding_results)
        
        final_assessment = {
            'necessity_score': total_necessity_score,
            'max_score': max_total_score,
            'necessity_percentage': necessity_percentage,
            'recommendations': recommendations
        }
        
        print(f"  综合必要性得分: {total_necessity_score}/{max_total_score} ({necessity_percentage:.1f}%)")
        print(f"  总体建议: {recommendations['overall_recommendation']}")
        print(f"  优先级: {recommendations['priority']}")
        
        embedding_results['final_assessment'] = final_assessment
        self.results['embedding_necessity_analysis'] = embedding_results
        
        return embedding_results

    def _suggest_embedding_architecture(self, clustering_quality, distance_percentile):
        """基于分析结果建议embedding架构"""
        suggestions = []
        
        if clustering_quality > 0.5:
            suggestions.append("考虑分层嵌入 (Hierarchical Embedding)")
        
        if distance_percentile > 0.8:
            suggestions.append("使用较大的嵌入维度 (64-128维)")
        elif distance_percentile > 0.5:
            suggestions.append("使用中等嵌入维度 (32-64维)")
        else:
            suggestions.append("使用较小的嵌入维度 (16-32维)")
        
        return suggestions

    def _generate_embedding_recommendations(self, necessity_percentage, embedding_results):
        """生成Subject Embedding建议"""
        
        if necessity_percentage > 75:
            overall_recommendation = "强烈推荐Subject Embedding"
            priority = "高"
            specific_methods = [
                "可学习的受试者嵌入层",
                "年龄信息融合",
                "对抗训练去偏差",
                "多层次嵌入架构"
            ]
        elif necessity_percentage > 50:
            overall_recommendation = "建议使用Subject Embedding"
            priority = "中"
            specific_methods = [
                "简单的受试者嵌入层",
                "年龄协变量控制",
                "正则化嵌入"
            ]
        elif necessity_percentage > 25:
            overall_recommendation = "可以尝试Subject Embedding"
            priority = "低"
            specific_methods = [
                "轻量级嵌入",
                "年龄校正",
                "特征标准化"
            ]
        else:
            overall_recommendation = "暂不推荐Subject Embedding"
            priority = "无"
            specific_methods = [
                "数据质量检查",
                "其他域适应方法",
                "简单年龄校正"
            ]
        
        return {
            'overall_recommendation': overall_recommendation,
            'priority': priority,
            'specific_methods': specific_methods,
            'estimated_improvement': self._estimate_embedding_improvement(necessity_percentage)
        }

    def _estimate_embedding_improvement(self, necessity_percentage):
        """估算Subject Embedding的预期改善"""
        if necessity_percentage > 75:
            return "0.08-0.15 F1提升"
        elif necessity_percentage > 50:
            return "0.05-0.10 F1提升"
        elif necessity_percentage > 25:
            return "0.02-0.05 F1提升"
        else:
            return "0.00-0.02 F1提升"

    def generate_comprehensive_visualizations(self):
        """生成完整的可视化分析图表"""
        print("\n" + "="*80)
        print("📊 Phase 7: 生成综合可视化分析")
        print("="*80)
        
        # 创建大型综合图表
        fig, axes = plt.subplots(4, 4, figsize=(24, 20))
        fig.suptitle('Subject Variability Analysis - Comprehensive Report', fontsize=20, fontweight='bold')
        
        # 1. LOSO性能分布
        if 'loso_analysis' in self.results and 'DeepLearning' in self.results['loso_analysis']:
            ax = axes[0, 0]
            dl_results = self.results['loso_analysis']['DeepLearning']
            individual_results = dl_results['individual_results']
            
            f1_scores = [r['f1_macro'] for r in individual_results]
            ages = [r['age'] for r in individual_results if r['age'] is not None]
            
            # ✅ 确保长度一致的检查
            valid_results = [r for r in individual_results if r['age'] is not None]
            if len(valid_results) > 0:
                ages = [r['age'] for r in valid_results]
                f1_scores = [r['f1_macro'] for r in valid_results]
                
                scatter = ax.scatter(ages, f1_scores, alpha=0.7, s=60, c='blue', label='LOSO Results')
                
                # 添加38号受试者
                if dl_results['subject_38_result']:  # ✅ 改成dl_results
                    subject_38_result = dl_results['subject_38_result']
                    if subject_38_result['age'] is not None:
                        ax.scatter([subject_38_result['age']], [subject_38_result['f1_macro']], 
                                    color='red', s=150, marker='*', label='Subject 38', zorder=5)
                
                # 添加回归线
                z = np.polyfit(ages, f1_scores, 1)
                p = np.poly1d(z)
                ax.plot(ages, p(ages), "r--", alpha=0.8)
                
                ax.set_title('LOSO Performance vs Age', fontweight='bold')
                ax.set_xlabel('Age (years)')
                ax.set_ylabel('F1 Macro Score')
                ax.legend()
                ax.grid(True, alpha=0.3)
        
        # 2. 受试者性能分布直方图
        if 'loso_analysis' in self.results and 'DeepLearning' in self.results['loso_analysis']:  # ✅ 改成DeepLearning
            ax = axes[0, 1]
            dl_results = self.results['loso_analysis']['DeepLearning']  # ✅ 改成DeepLearning
            individual_results = dl_results['individual_results']
            
            f1_scores = [r['f1_macro'] for r in individual_results]
            
            ax.hist(f1_scores, bins=15, alpha=0.7, color='skyblue', edgecolor='black')
            ax.axvline(np.mean(f1_scores), color='blue', linestyle='--', linewidth=2, label=f'Mean: {np.mean(f1_scores):.3f}')
            
            if dl_results['subject_38_result']:  # ✅ 改成dl_results
                ax.axvline(dl_results['subject_38_result']['f1_macro'], color='red', linestyle='--', 
                            linewidth=2, label=f'Subject 38: {dl_results["subject_38_result"]["f1_macro"]:.3f}')
            
            ax.set_title('LOSO F1 Score Distribution', fontweight='bold')
            ax.set_xlabel('F1 Macro Score')
            ax.set_ylabel('Number of Subjects')
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        # 3. 年龄分布对比
        ax = axes[0, 2]
        subject_38_age = self.subject_stats_df[self.subject_stats_df['subject_id'] == 38]['age'].iloc[0]
        training_ages = self.subject_stats_df[self.subject_stats_df['subject_id'] != 38]['age'].values
        
        ax.hist(training_ages, bins=10, alpha=0.7, color='lightblue', label='Training Subjects (1-37)')
        ax.axvline(subject_38_age, color='red', linestyle='--', linewidth=3, label=f'Subject 38: {subject_38_age:.1f}')
        ax.axvline(np.mean(training_ages), color='blue', linestyle=':', linewidth=2, label=f'Training Mean: {np.mean(training_ages):.1f}')
        
        ax.set_title('Age Distribution Comparison', fontweight='bold')
        ax.set_xlabel('Age (years)')
        ax.set_ylabel('Number of Subjects')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 4. 38号异常性雷达图
        if 'subject_38_analysis' in self.results:
            ax = axes[0, 3]
            anomaly_analysis = self.results['subject_38_analysis']['anomaly_analysis']
            age_analysis = self.results['subject_38_analysis']['age_analysis']
            feature_analysis = self.results['subject_38_analysis']['feature_analysis']
            sample_analysis = self.results['subject_38_analysis']['sample_analysis']
            
            categories = ['Age\nAnomaly', 'Feature\nAnomaly', 'Sample\nDistance', 'Class\nDistribution']
            scores = [
                min(abs(age_analysis['age_z_score']) / 3, 1),
                feature_analysis['outlier_percentage_2sigma'] / 30,
                min(sample_analysis['distance_z_score'] / 3, 1),
                anomaly_analysis['anomaly_score'] / anomaly_analysis['max_possible_score']
            ]
            
            angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False)
            scores_plot = scores + [scores[0]]
            angles_plot = np.concatenate((angles, [angles[0]]))
            
            ax.plot(angles_plot, scores_plot, 'o-', linewidth=2, color='red', markersize=6)
            ax.fill(angles_plot, scores_plot, alpha=0.25, color='red')
            ax.set_xticks(angles)
            ax.set_xticklabels(categories)
            ax.set_ylim(0, 1)
            ax.set_title('Subject 38 Anomaly Profile', fontweight='bold')
            ax.grid(True)
        
        # 5. 受试者相似性热图
        if 'subject_similarity_analysis' in self.results:
            ax = axes[1, 0]
            similarity_results = self.results['subject_similarity_analysis']
            correlation_matrix = similarity_results['correlation_matrix']
            subject_ids = similarity_results['subject_ids']
            
            # 重新排序，把38号放在最后
            if 38 in subject_ids:
                subject_38_idx = subject_ids.index(38)
                reorder_indices = [i for i in range(len(subject_ids)) if i != subject_38_idx] + [subject_38_idx]
                correlation_reordered = correlation_matrix[np.ix_(reorder_indices, reorder_indices)]
                
                im = ax.imshow(correlation_reordered, cmap='RdBu_r', vmin=-1, vmax=1)
                ax.set_title('Subject Similarity Matrix', fontweight='bold')
                
                # 高亮38号
                ax.axhline(len(subject_ids)-1, color='red', linewidth=2)
                ax.axvline(len(subject_ids)-1, color='red', linewidth=2)
                
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        
        # 6. 聚类树状图
        if 'subject_similarity_analysis' in self.results:
            ax = axes[1, 1]
            similarity_results = self.results['subject_similarity_analysis']
            linkage_matrix = similarity_results['clustering']['linkage_matrix']
            subject_ids = similarity_results['subject_ids']
            
            dendrogram(linkage_matrix, ax=ax, labels=[f'S{sid}' for sid in subject_ids], 
                        leaf_rotation=90, leaf_font_size=8)
            ax.set_title('Subject Hierarchical Clustering', fontweight='bold')
            ax.set_xlabel('Subjects')
            ax.set_ylabel('Distance')
        
        # 7. 模型性能对比
        if 'loso_analysis' in self.results:
            ax = axes[1, 2]
            
            models = list(self.results['loso_analysis'].keys())
            loso_means = []
            subject_38_scores = []
            
            for model in models:
                results = self.results['loso_analysis'][model]
                loso_means.append(results['mean_f1_macro'])
                if results['subject_38_result']:
                    subject_38_scores.append(results['subject_38_result']['f1_macro'])
                else:
                    subject_38_scores.append(0)
            
            x = np.arange(len(models))
            width = 0.35
            
            bars1 = ax.bar(x - width/2, loso_means, width, label='LOSO Mean', alpha=0.8, color='lightblue')
            bars2 = ax.bar(x + width/2, subject_38_scores, width, label='Subject 38', alpha=0.8, color='lightcoral')
            
            ax.set_title('Model Performance Comparison', fontweight='bold')
            ax.set_xlabel('Models')
            ax.set_ylabel('F1 Macro Score')
            ax.set_xticks(x)
            ax.set_xticklabels(models, rotation=45)
            ax.legend()
            ax.grid(True, alpha=0.3, axis='y')
            
            # 添加数值标签
            for bars in [bars1, bars2]:
                for bar in bars:
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                            f'{height:.3f}', ha='center', va='bottom', fontsize=9)
        
        # 8. 年龄-性能回归分析
        if 'age_effect_analysis' in self.results and 'age_performance' in self.results['age_effect_analysis']:
            ax = axes[1, 3]
            age_perf = self.results['age_effect_analysis']['age_performance']
            
            # 重新绘制年龄-性能散点图和回归线
            if 'loso_analysis' in self.results and 'DeepLearning' in self.results['loso_analysis']:
                dl_results = self.results['loso_analysis']['DeepLearning']
                individual_results = dl_results['individual_results']
                
                ages = [r['age'] for r in individual_results if r['age'] is not None]
                f1_scores = [r['f1_macro'] for r in individual_results if r['age'] is not None]
                
                if len(ages) > 0:  # 确保有有效数据
                    ax.scatter(ages, f1_scores, alpha=0.6, s=50, color='blue', label='Training Subjects')
                    
                    # 回归线
                    age_range = np.linspace(min(ages), max(ages), 100)
                    regression_line = age_perf['age_slope'] * age_range + age_perf['age_intercept']
                    ax.plot(age_range, regression_line, 'r-', linewidth=2, label='Age Trend')
                
                # 38号受试者预测vs实际
                if age_perf['actual_38_performance']:
                    subject_38_age = self.subject_stats_df[self.subject_stats_df['subject_id'] == 38]['age'].iloc[0]
                    ax.scatter([subject_38_age], [age_perf['predicted_38_performance']], 
                                color='orange', s=100, marker='^', label='Predicted 38')
                    ax.scatter([subject_38_age], [age_perf['actual_38_performance']], 
                                color='red', s=100, marker='*', label='Actual 38')
                
                ax.set_title(f'Age-Performance Relationship\n(r={age_perf["age_performance_correlation"]:.3f})', fontweight='bold')
                ax.set_xlabel('Age (years)')
                ax.set_ylabel('F1 Macro Score')
                ax.legend()
                ax.grid(True, alpha=0.3)
        
        # 9. 特征异常分析
        if 'subject_38_analysis' in self.results:
            ax = axes[2, 0]
            feature_analysis = self.results['subject_38_analysis']['feature_analysis']
            feature_z_scores = feature_analysis['feature_z_scores']
            
            ax.plot(np.abs(feature_z_scores), alpha=0.7, linewidth=1)
            ax.axhline(2, color='orange', linestyle='--', label='2σ threshold')
            ax.axhline(3, color='red', linestyle='--', label='3σ threshold')
            
            # 标记最异常的特征
            most_outlier_features = feature_analysis['most_outlier_features']
            ax.scatter(most_outlier_features, np.abs(feature_z_scores)[most_outlier_features], 
                        color='red', s=30, zorder=5)
            
            ax.set_title(f'Feature Anomaly Profile\n({feature_analysis["outlier_percentage_2sigma"]:.1f}% outliers)', fontweight='bold')
            ax.set_xlabel('Feature Index')
            ax.set_ylabel('|Z-Score|')
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        # 10. 样本距离分布
        if 'subject_38_analysis' in self.results:
            ax = axes[2, 1]
            sample_analysis = self.results['subject_38_analysis']['sample_analysis']
            
            # 模拟距离分布（实际实现中需要保存这些数据）
            training_mean = sample_analysis['mean_distance_training']
            training_std = sample_analysis['std_distance_training']
            subject_38_mean = sample_analysis['mean_distance_38']
            
            # 绘制分布比较
            x = np.linspace(0, max(training_mean + 3*training_std, subject_38_mean + 0.1), 100)
            training_dist = stats.norm.pdf(x, training_mean, training_std)
            
            ax.plot(x, training_dist, 'b-', linewidth=2, label='Training Distribution')
            ax.axvline(subject_38_mean, color='red', linestyle='--', linewidth=2, 
                        label=f'Subject 38 Mean: {subject_38_mean:.3f}')
            ax.axvline(training_mean, color='blue', linestyle=':', linewidth=2, 
                        label=f'Training Mean: {training_mean:.3f}')
            
            ax.set_title('Sample Distance Distribution', fontweight='bold')
            ax.set_xlabel('Mean Distance to Neighbors')
            ax.set_ylabel('Density')
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        # 11. Subject Embedding必要性评估
        if 'embedding_necessity_analysis' in self.results:
            ax = axes[2, 2]
            embedding_results = self.results['embedding_necessity_analysis']
            final_assessment = embedding_results['final_assessment']
            
            # 创建必要性条形图
            categories = ['LOSO\nVariability', 'Age\nEffect', 'Anomaly\nLevel', 'Overall']
            scores = []
            
            if 'loso_necessity' in embedding_results:
                loso_score = embedding_results['loso_necessity']['necessity_score'] / embedding_results['loso_necessity']['max_score']
                scores.append(loso_score)
            else:
                scores.append(0)
            
            # 年龄效应得分（简化）
            if 'age_effect_analysis' in self.results:
                age_gap = abs(self.results['age_effect_analysis']['age_distribution']['age_gap'])
                age_score = min(age_gap / 10, 1)
                scores.append(age_score)
            else:
                scores.append(0)
            
            # 异常性得分
            if 'subject_38_analysis' in self.results:
                anomaly_score = self.results['subject_38_analysis']['anomaly_analysis']['anomaly_score']
                max_anomaly = self.results['subject_38_analysis']['anomaly_analysis']['max_possible_score']
                scores.append(anomaly_score / max_anomaly)
            else:
                scores.append(0)
            
            # 总体得分
            scores.append(final_assessment['necessity_percentage'] / 100)
            
            colors = ['lightblue', 'lightgreen', 'orange', 'red']
            bars = ax.bar(categories, scores, color=colors, alpha=0.7, edgecolor='black')
            
            ax.set_title('Subject Embedding Necessity', fontweight='bold')
            ax.set_ylabel('Necessity Score (0-1)')
            ax.set_ylim(0, 1)
            
            # 添加数值标签
            for bar, score in zip(bars, scores):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                        f'{score:.2f}', ha='center', va='bottom', fontweight='bold')
            
            ax.grid(True, alpha=0.3, axis='y')
        
        # 12. 解决方案优先级
        ax = axes[2, 3]
        
        # 基于所有分析结果生成解决方案建议
        solutions = []
        priorities = []
        
        # Subject Embedding
        if 'embedding_necessity_analysis' in self.results:
            necessity = self.results['embedding_necessity_analysis']['final_assessment']['necessity_percentage']
            solutions.append('Subject\nEmbedding')
            priorities.append(necessity / 100)
        
        # 年龄校正
        if 'age_effect_analysis' in self.results:
            age_gap = abs(self.results['age_effect_analysis']['age_distribution']['age_gap'])
            age_priority = min(age_gap / 10, 1)
            solutions.append('Age\nCorrection')
            priorities.append(age_priority)
        
        # 数据质量检查
        if 'subject_38_analysis' in self.results:
            anomaly_score = self.results['subject_38_analysis']['anomaly_analysis']['anomaly_score']
            max_anomaly = self.results['subject_38_analysis']['anomaly_analysis']['max_possible_score']
            data_quality_priority = anomaly_score / max_anomaly
            solutions.append('Data Quality\nCheck')
            priorities.append(data_quality_priority)
        
        # 域适应
        if 'loso_analysis' in self.results:
            dl_results = self.results['loso_analysis'].get('DeepLearning')
            if dl_results:
                loso_std = dl_results['std_f1_macro']
                loso_mean = dl_results['mean_f1_macro']
                variability = loso_std / loso_mean if loso_mean > 0 else 0
                domain_priority = min(variability * 2, 1)
                solutions.append('Domain\nAdaptation')
                priorities.append(domain_priority)
        
        if solutions and priorities:
            colors = plt.cm.RdYlGn_r(np.array(priorities))
            bars = ax.barh(solutions, priorities, color=colors, alpha=0.8, edgecolor='black')
            
            ax.set_title('Solution Priority Ranking', fontweight='bold')
            ax.set_xlabel('Priority Score (0-1)')
            ax.set_xlim(0, 1)
            
            # 添加数值标签
            for bar, priority in zip(bars, priorities):
                width = bar.get_width()
                ax.text(width + 0.02, bar.get_y() + bar.get_height()/2.,
                        f'{priority:.2f}', ha='left', va='center', fontweight='bold')
            
            ax.grid(True, alpha=0.3, axis='x')
        
        # 13-16. 详细统计信息面板
        for i, (row, col) in enumerate([(3, 0), (3, 1), (3, 2), (3, 3)]):
            ax = axes[row, col]
            ax.axis('off')
            
            if 'loso_analysis' in self.results and 'DeepLearning' in self.results['loso_analysis']:
                dl_results = self.results['loso_analysis']['DeepLearning']
                
                stats_text = "LOSO Analysis Summary\n" + "="*25 + "\n\n"
                stats_text += f"Subjects tested: {dl_results['n_subjects_tested']}\n"
                stats_text += f"Mean F1: {dl_results['mean_f1_macro']:.4f}\n"
                stats_text += f"Std F1: {dl_results['std_f1_macro']:.4f}\n"
                stats_text += f"Range: [{dl_results['min_f1_macro']:.4f}, {dl_results['max_f1_macro']:.4f}]\n"
                
                if dl_results['subject_38_result']:
                    subject_38_f1 = dl_results['subject_38_result']['f1_macro']
                    gap = dl_results['mean_f1_macro'] - subject_38_f1
                    stats_text += f"\nSubject 38 F1: {subject_38_f1:.4f}\n"
                    stats_text += f"Performance gap: {gap:+.4f}\n"
                    stats_text += f"Gap in std: {gap/dl_results['std_f1_macro']:.1f}σ\n"
                    
                    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=10,
                            verticalalignment='top', fontfamily='monospace',
                            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.8))
            
            elif i == 1:  # 年龄统计
                if 'age_effect_analysis' in self.results:
                    age_dist = self.results['age_effect_analysis']['age_distribution']
                    
                    stats_text = "Age Analysis Summary\n" + "="*22 + "\n\n"
                    stats_text += f"Training mean: {age_dist['training_mean']:.1f} ± {age_dist['training_std']:.1f}\n"
                    stats_text += f"Training range: [{age_dist['training_range'][0]:.1f}, {age_dist['training_range'][1]:.1f}]\n"
                    stats_text += f"Subject 38: {age_dist['subject_38_age']:.1f} years\n"
                    stats_text += f"Age gap: {age_dist['age_gap']:+.1f} years\n"
                    stats_text += f"Z-score: {age_dist['age_z_score']:+.2f}\n"
                    
                    if 'age_performance' in self.results['age_effect_analysis']:
                        age_perf = self.results['age_effect_analysis']['age_performance']
                        stats_text += f"\nAge-performance correlation: {age_perf['age_performance_correlation']:.3f}\n"
                        if age_perf['actual_38_performance']:
                            residual = age_perf['actual_38_performance'] - age_perf['predicted_38_performance']
                            stats_text += f"Age-corrected residual: {residual:+.4f}\n"
                    
                    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=10,
                            verticalalignment='top', fontfamily='monospace',
                            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgreen', alpha=0.8))
            
            elif i == 2:  # 异常性统计
                if 'subject_38_analysis' in self.results:
                    anomaly = self.results['subject_38_analysis']['anomaly_analysis']
                    feature_anom = self.results['subject_38_analysis']['feature_analysis']
                    
                    stats_text = "Subject 38 Anomaly\n" + "="*18 + "\n\n"
                    stats_text += f"Anomaly score: {anomaly['anomaly_score']}/{anomaly['max_possible_score']}\n"
                    stats_text += f"Outlier features (2σ): {feature_anom['outlier_percentage_2sigma']:.1f}%\n"
                    stats_text += f"Outlier features (3σ): {feature_anom['outlier_percentage_3sigma']:.1f}%\n"
                    stats_text += f"Max |Z-score|: {feature_anom['max_z_score']:.2f}\n"
                    
                    stats_text += f"\nAnomaly reasons:\n"
                    for reason in anomaly['anomaly_reasons'][:3]:  # 最多显示3个
                        stats_text += f"• {reason}\n"
                    
                    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=10,
                            verticalalignment='top', fontfamily='monospace',
                            bbox=dict(boxstyle='round,pad=0.5', facecolor='orange', alpha=0.8))
            
            elif i == 3:  # 建议总结
                if 'embedding_necessity_analysis' in self.results:
                    final_assessment = self.results['embedding_necessity_analysis']['final_assessment']
                    
                    stats_text = "Recommendations\n" + "="*15 + "\n\n"
                    stats_text += f"Overall: {final_assessment['recommendations']['overall_recommendation']}\n"
                    stats_text += f"Priority: {final_assessment['recommendations']['priority']}\n"
                    stats_text += f"Expected improvement: {final_assessment['recommendations']['estimated_improvement']}\n"
                    
                    stats_text += f"\nSpecific methods:\n"
                    for method in final_assessment['recommendations']['specific_methods'][:3]:
                        stats_text += f"• {method}\n"
                    
                    stats_text += f"\nNecessity score: {final_assessment['necessity_percentage']:.1f}%\n"
                    
                    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=10,
                            verticalalignment='top', fontfamily='monospace',
                            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightcoral', alpha=0.8))
        
        plt.tight_layout()
        viz_path = os.path.join(self.save_path, 'visualizations', 'comprehensive_subject_analysis.png')
        plt.savefig(viz_path, dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"✅ 综合可视化已保存: {viz_path}")
        
        # 生成单独的高质量图表
        self._generate_individual_plots()

    def _generate_individual_plots(self):
        """生成单独的高质量图表"""
        print("🔄 正在生成单独的高质量图表...")
        
        # 1. LOSO详细分析图
        if 'loso_analysis' in self.results:
            self._plot_loso_detailed_analysis()
        
        # 2. 38号受试者异常性详细分析
        if 'subject_38_analysis' in self.results:
            self._plot_subject_38_detailed_analysis()
        
        # 3. 受试者相似性详细分析
        if 'subject_similarity_analysis' in self.results:
            self._plot_similarity_detailed_analysis()
        
        print("✅ 单独图表生成完成")

    def _plot_loso_detailed_analysis(self):
        """LOSO详细分析图"""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Leave-One-Subject-Out Detailed Analysis', fontsize=16, fontweight='bold')
        
        dl_results = self.results['loso_analysis']['DeepLearning']
        individual_results = dl_results['individual_results']
        
        # 提取数据
        subject_ids = [r['subject_id'] for r in individual_results]
        f1_scores = [r['f1_macro'] for r in individual_results]
        ages = [r['age'] for r in individual_results if r['age'] is not None]  # 处理None值
        accuracies = [r['accuracy'] for r in individual_results]
        
        # 1. 受试者性能排序
        ax = axes[0, 0]
        sorted_indices = np.argsort(f1_scores)
        
        bars = ax.bar(range(len(f1_scores)), [f1_scores[i] for i in sorted_indices], 
                        color='lightblue', alpha=0.7, edgecolor='black')
        
        # 标注最好和最差的受试者
        worst_idx = sorted_indices[0]
        best_idx = sorted_indices[-1]
        bars[0].set_color('red')
        bars[-1].set_color('green')
        
        ax.set_title('LOSO Performance Ranking')
        ax.set_xlabel('Subject Rank')
        ax.set_ylabel('F1 Macro Score')
        ax.grid(True, alpha=0.3, axis='y')
        
        # 添加38号受试者对比线
        if dl_results['subject_38_result']:  # ✅ 改成 dl_results
            ax.axhline(dl_results['subject_38_result']['f1_macro'], color='red', 
                        linestyle='--', linewidth=2, label='Subject 38')
            ax.legend()
        
        # 2. 年龄vs性能散点图（详细版）
        ax = axes[0, 1]
        scatter = ax.scatter(ages, f1_scores, c=ages, cmap='viridis', s=80, alpha=0.7, edgecolors='black')
        
        # 添加回归线
        z = np.polyfit(ages, f1_scores, 1)
        p = np.poly1d(z)
        ax.plot(ages, p(ages), "r--", alpha=0.8, linewidth=2)
        
        # 添加相关系数
        correlation = np.corrcoef(ages, f1_scores)[0, 1]
        ax.text(0.05, 0.95, f'r = {correlation:.3f}', transform=ax.transAxes, 
                bbox=dict(boxstyle="round", facecolor='white', alpha=0.8))
        
        ax.set_title('Age vs LOSO Performance')
        ax.set_xlabel('Age (years)')
        ax.set_ylabel('F1 Macro Score')
        plt.colorbar(scatter, ax=ax, label='Age')
        ax.grid(True, alpha=0.3)
        
        # 3. 性能分布和统计
        ax = axes[1, 0]
        
        # 直方图
        n, bins, patches = ax.hist(f1_scores, bins=12, alpha=0.7, color='skyblue', 
                                    edgecolor='black', density=True)
        
        # 添加统计线
        mean_f1 = np.mean(f1_scores)
        std_f1 = np.std(f1_scores)
        ax.axvline(mean_f1, color='blue', linestyle='-', linewidth=2, label=f'Mean: {mean_f1:.3f}')
        ax.axvline(mean_f1 + std_f1, color='orange', linestyle='--', linewidth=2, label=f'+1σ: {mean_f1+std_f1:.3f}')
        ax.axvline(mean_f1 - std_f1, color='orange', linestyle='--', linewidth=2, label=f'-1σ: {mean_f1-std_f1:.3f}')
        
        # 添加38号受试者
        if dl_results['subject_38_result']:
            ax.axhline(dl_results['subject_38_result']['f1_macro'], color='red', 
                        linestyle='--', linewidth=2, label='Subject 38')
            ax.legend()
        
        ax.set_title('LOSO F1 Distribution & Statistics')
        ax.set_xlabel('F1 Macro Score')
        ax.set_ylabel('Density')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 4. 多模型对比
        ax = axes[1, 1]
        
        models = list(self.results['loso_analysis'].keys())
        loso_means = []
        loso_stds = []
        subject_38_scores = []
        
        for model in models:
            results = self.results['loso_analysis'][model]
            loso_means.append(results['mean_f1_macro'])
            loso_stds.append(results['std_f1_macro'])
            if results['subject_38_result']:
                subject_38_scores.append(results['subject_38_result']['f1_macro'])
            else:
                subject_38_scores.append(0)
        
        x = np.arange(len(models))
        width = 0.35
        
        # 误差条形图
        bars1 = ax.bar(x - width/2, loso_means, width, yerr=loso_stds, 
                        label='LOSO Mean ± Std', alpha=0.8, color='lightblue', 
                        capsize=5, error_kw={'elinewidth': 2})
        bars2 = ax.bar(x + width/2, subject_38_scores, width, 
                        label='Subject 38', alpha=0.8, color='lightcoral')
        
        ax.set_title('Multi-Model Performance Comparison')
        ax.set_xlabel('Models')
        ax.set_ylabel('F1 Macro Score')
        ax.set_xticks(x)
        ax.set_xticklabels(models, rotation=45)
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.save_path, 'visualizations', 'loso_detailed_analysis.png'), 
                    dpi=300, bbox_inches='tight')
        plt.close()

    def _plot_subject_38_detailed_analysis(self):
        """38号受试者详细异常分析图"""
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('Subject 38 Detailed Anomaly Analysis', fontsize=16, fontweight='bold')
        
        anomaly_results = self.results['subject_38_analysis']
        
        # 1. 年龄对比
        ax = axes[0, 0]
        age_analysis = anomaly_results['age_analysis']
        
        training_ages = self.subject_stats_df[self.subject_stats_df['subject_id'] != 38]['age'].values
        subject_38_age = age_analysis['subject_38_age']
        
        ax.hist(training_ages, bins=10, alpha=0.7, color='lightblue', 
                label=f'Training (n={len(training_ages)})', density=True)
        ax.axvline(subject_38_age, color='red', linestyle='--', linewidth=3, 
                    label=f'Subject 38: {subject_38_age:.1f}')
        ax.axvline(np.mean(training_ages), color='blue', linestyle=':', linewidth=2, 
                    label=f'Training Mean: {np.mean(training_ages):.1f}')
        
        ax.set_title(f'Age Distribution\n(Z-score: {age_analysis["age_z_score"]:+.2f})')
        ax.set_xlabel('Age (years)')
        ax.set_ylabel('Density')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 2. 特征异常热图
        ax = axes[0, 1]
        feature_analysis = anomaly_results['feature_analysis']
        feature_z_scores = feature_analysis['feature_z_scores']
        
        # 重塑为2D热图（如果特征数是完全平方数，否则补零）
        n_features = len(feature_z_scores)
        grid_size = int(np.ceil(np.sqrt(n_features)))
        z_scores_padded = np.pad(feature_z_scores, (0, grid_size**2 - n_features), 'constant')
        z_scores_2d = z_scores_padded.reshape(grid_size, grid_size)
        
        im = ax.imshow(z_scores_2d, cmap='RdBu_r', vmin=-5, vmax=5)
        ax.set_title(f'Feature Z-Scores Heatmap\n({feature_analysis["outlier_percentage_2sigma"]:.1f}% outliers)')
        plt.colorbar(im, ax=ax, label='Z-Score')
        
        # 3. 特征异常分布
        ax = axes[0, 2]
        
        ax.hist(np.abs(feature_z_scores), bins=30, alpha=0.7, color='orange', 
                edgecolor='black', density=True)
        ax.axvline(2, color='orange', linestyle='--', linewidth=2, label='2σ threshold')
        ax.axvline(3, color='red', linestyle='--', linewidth=2, label='3σ threshold')
        ax.axvline(np.mean(np.abs(feature_z_scores)), color='blue', linestyle='-', 
                    linewidth=2, label=f'Mean: {np.mean(np.abs(feature_z_scores)):.2f}')
        
        ax.set_title('Feature |Z-Score| Distribution')
        ax.set_xlabel('|Z-Score|')
        ax.set_ylabel('Density')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 4. 最异常特征
        ax = axes[1, 0]
        most_outlier_features = feature_analysis['most_outlier_features']
        outlier_z_scores = feature_z_scores[most_outlier_features]
        
        colors = ['red' if z > 0 else 'blue' for z in outlier_z_scores]
        bars = ax.bar(range(len(outlier_z_scores)), outlier_z_scores, 
                        color=colors, alpha=0.7, edgecolor='black')
        
        ax.set_title('Top 10 Most Outlier Features')
        ax.set_xlabel('Feature Rank')
        ax.set_ylabel('Z-Score')
        ax.set_xticks(range(len(most_outlier_features)))
        ax.set_xticklabels([f'F{f}' for f in most_outlier_features], rotation=45)
        ax.grid(True, alpha=0.3, axis='y')
        ax.axhline(0, color='black', linestyle='-', alpha=0.5)
        
        # 5. 样本距离分析（模拟）
        ax = axes[1, 1]
        sample_analysis = anomaly_results['sample_analysis']
        
        # 模拟距离分布
        training_mean = sample_analysis['mean_distance_training']
        training_std = sample_analysis['std_distance_training']
        subject_38_mean = sample_analysis['mean_distance_38']
        
        x = np.linspace(max(0, training_mean - 3*training_std), 
                        training_mean + 4*training_std, 100)
        training_dist = stats.norm.pdf(x, training_mean, training_std)
        
        ax.fill_between(x, training_dist, alpha=0.5, color='lightblue', label='Training Distribution')
        ax.axvline(training_mean, color='blue', linestyle=':', linewidth=2, 
                    label=f'Training Mean: {training_mean:.3f}')
        ax.axvline(subject_38_mean, color='red', linestyle='--', linewidth=3, 
                    label=f'Subject 38: {subject_38_mean:.3f}')
        
        # 添加Z分数注释
        z_score = sample_analysis['distance_z_score']
        ax.text(0.05, 0.95, f'Z-score: {z_score:+.2f}', transform=ax.transAxes,
                bbox=dict(boxstyle="round", facecolor='white', alpha=0.8))
        
        ax.set_title('Sample Distance Distribution')
        ax.set_xlabel('Mean Distance to Neighbors')
        ax.set_ylabel('Density')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 6. 综合异常性雷达图
        ax = axes[1, 2]
        
        categories = ['Age\nAnomaly', 'Feature\nOutliers', 'Sample\nDistance', 
                        'Class\nDistribution', 'Overall\nAnomaly']
        
        scores = [
            min(abs(age_analysis['age_z_score']) / 3, 1),
            feature_analysis['outlier_percentage_2sigma'] / 30,
            min(abs(sample_analysis['distance_z_score']) / 3, 1),
            anomaly_results['class_analysis']['kl_divergence'] / 2,
            anomaly_results['anomaly_analysis']['anomaly_score'] / anomaly_results['anomaly_analysis']['max_possible_score']
        ]
        
        # 雷达图
        angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False)
        scores_plot = scores + [scores[0]]
        angles_plot = np.concatenate((angles, [angles[0]]))
        
        ax.plot(angles_plot, scores_plot, 'o-', linewidth=2, color='red', markersize=8)
        ax.fill(angles_plot, scores_plot, alpha=0.25, color='red')
        ax.set_xticks(angles)
        ax.set_xticklabels(categories)
        ax.set_ylim(0, 1)
        ax.set_title('Subject 38 Anomaly Profile')
        ax.grid(True)
        
        # 添加数值标签
        for angle, score, category in zip(angles, scores, categories):
            ax.text(angle, score + 0.05, f'{score:.2f}', ha='center', va='center',
                    fontweight='bold', color='red')
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.save_path, 'visualizations', 'subject_38_detailed_analysis.png'), 
                    dpi=300, bbox_inches='tight')
        plt.close()

    def _plot_similarity_detailed_analysis(self):
        """受试者相似性详细分析图"""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Subject Similarity Detailed Analysis', fontsize=16, fontweight='bold')
        
        similarity_results = self.results['subject_similarity_analysis']
        
        # 1. 相似性矩阵（重新排序突出38号）
        ax = axes[0, 0]
        correlation_matrix = similarity_results['correlation_matrix']
        subject_ids = similarity_results['subject_ids']
        
        if 38 in subject_ids:
            # 重新排序，38号放最后
            subject_38_idx = subject_ids.index(38)
            reorder_indices = [i for i in range(len(subject_ids)) if i != subject_38_idx] + [subject_38_idx]
            correlation_reordered = correlation_matrix[np.ix_(reorder_indices, reorder_indices)]
            
            im = ax.imshow(correlation_reordered, cmap='RdBu_r', vmin=-1, vmax=1)
            
            # 高亮38号
            n_subjects = len(subject_ids)
            ax.axhline(n_subjects-1, color='red', linewidth=3, alpha=0.8)
            ax.axvline(n_subjects-1, color='red', linewidth=3, alpha=0.8)
            
            # 添加边框
            for spine in ax.spines.values():
                spine.set_edgecolor('red')
                spine.set_linewidth(2)
            
            ax.set_title('Subject Correlation Matrix\n(Subject 38 highlighted)')
            plt.colorbar(im, ax=ax, label='Correlation')
        
        # 2. 聚类树状图（详细版）
        ax = axes[0, 1]
        linkage_matrix = similarity_results['clustering']['linkage_matrix']
        
        dendrogram(linkage_matrix, ax=ax, 
                    labels=[f'S{sid}' for sid in subject_ids],
                    leaf_rotation=90, leaf_font_size=10,
                    color_threshold=0.7*max(linkage_matrix[:, 2]))
        
        ax.set_title('Subject Hierarchical Clustering')
        ax.set_xlabel('Subjects')
        ax.set_ylabel('Distance')
        
        # 3. 聚类质量评估
        ax = axes[1, 0]
        clustering = similarity_results['clustering']
        
        n_clusters_range = range(2, len(clustering['silhouette_scores']) + 2)
        silhouette_scores = clustering['silhouette_scores']
        
        ax.plot(n_clusters_range, silhouette_scores, 'o-', linewidth=2, markersize=8)
        ax.axvline(clustering['best_n_clusters'], color='red', linestyle='--', 
                    linewidth=2, label=f'Best: {clustering["best_n_clusters"]} clusters')
        ax.axhline(clustering['best_silhouette'], color='red', linestyle=':', 
                    alpha=0.7, label=f'Best score: {clustering["best_silhouette"]:.3f}')
        
        ax.set_title('Clustering Quality (Silhouette Score)')
        ax.set_xlabel('Number of Clusters')
        ax.set_ylabel('Silhouette Score')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 4. 年龄vs相似性关系
        ax = axes[1, 1]
        
        ages = similarity_results['ages']
        distance_matrix = similarity_results['distance_matrix']
        
        # 计算年龄差异与特征距离的关系
        age_diffs = []
        feature_distances = []
        
        for i in range(len(subject_ids)):
            for j in range(i+1, len(subject_ids)):
                age_diff = abs(ages[i] - ages[j])
                feature_distance = distance_matrix[i, j]
                age_diffs.append(age_diff)
                feature_distances.append(feature_distance)
        
        # 散点图
        scatter = ax.scatter(age_diffs, feature_distances, alpha=0.6, s=30)
        
        # 回归线
        z = np.polyfit(age_diffs, feature_distances, 1)
        p = np.poly1d(z)
        age_range = np.linspace(min(age_diffs), max(age_diffs), 100)
        ax.plot(age_range, p(age_range), "r--", alpha=0.8, linewidth=2)
        
        # 相关系数
        correlation = similarity_results['age_distance_correlation']
        ax.text(0.05, 0.95, f'r = {correlation:.3f}', transform=ax.transAxes,
                bbox=dict(boxstyle="round", facecolor='white', alpha=0.8))
        
        # 高亮38号相关的点（如果能识别）
        if 38 in subject_ids:
            subject_38_idx = subject_ids.index(38)
            subject_38_age = ages[subject_38_idx]
            
            # 找到涉及38号的点
            for i, (age_diff, feat_dist) in enumerate(zip(age_diffs, feature_distances)):
                # 这需要更复杂的逻辑来确定哪些点涉及38号
                pass
        
        ax.set_title('Age Difference vs Feature Distance')
        ax.set_xlabel('Age Difference (years)')
        ax.set_ylabel('Feature Distance')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.save_path, 'visualizations', 'similarity_detailed_analysis.png'), 
                    dpi=300, bbox_inches='tight')
        plt.close()

    def generate_comprehensive_report(self):
        """
        生成完整的分析报告
        """
        print("\n" + "="*80)
        print("📝 Phase 8: 生成综合分析报告")
        print("="*80)
        
        report_path = os.path.join(self.save_path, 'reports', 'subject_variability_analysis_report.md')
        
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("# Subject Variability Analysis Report\n\n")
            f.write(f"**生成时间**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            # 执行摘要
            f.write("## 执行摘要\n\n")
            self._write_executive_summary(f)
            
            # 数据概况
            f.write("## 数据概况\n\n")
            self._write_data_overview(f)
            
            # LOSO分析结果
            f.write("## Leave-One-Subject-Out (LOSO) 分析\n\n")
            self._write_loso_analysis(f)
            
            # 38号受试者分析
            f.write("## Subject 38 异常性分析\n\n")
            self._write_subject_38_analysis(f)
            
            # 受试者相似性分析
            f.write("## 受试者相似性分析\n\n")
            self._write_similarity_analysis(f)
            
            # 年龄效应分析
            f.write("## 年龄效应分析\n\n")
            self._write_age_effect_analysis(f)
            
            # Subject Embedding建议
            f.write("## Subject Embedding 建议\n\n")
            self._write_embedding_recommendations(f)
            
            # 实施路线图
            f.write("## 实施路线图\n\n")
            self._write_implementation_roadmap(f)
        
        # 保存结果到JSON
        results_path = os.path.join(self.save_path, 'reports', 'analysis_results.json')
        with open(results_path, 'w', encoding='utf-8') as f:
            # 转换numpy数组为列表以便JSON序列化
            json_results = self._prepare_results_for_json()
            json.dump(json_results, f, indent=2, ensure_ascii=False)
        
        print(f"✅ 综合分析报告已保存:")
        print(f"  - Markdown报告: {report_path}")
        print(f"  - JSON结果: {results_path}")
        
        return report_path, results_path

    def _write_executive_summary(self, f):
        """写入执行摘要"""
        f.write("本次分析深入评估了受试者个体差异对模型泛化性能的影响，并提供了针对性的解决方案建议。\n\n")
        
        # 关键发现
        if 'loso_analysis' in self.results and 'DeepLearning' in self.results['loso_analysis']:
            dl_results = self.results['loso_analysis']['DeepLearning']
            loso_mean = dl_results['mean_f1_macro']
            if dl_results['subject_38_result']:
                subject_38_f1 = dl_results['subject_38_result']['f1_macro']
                gap = loso_mean - subject_38_f1
                
                f.write("### 关键发现\n\n")
                f.write(f"- **LOSO平均性能**: {loso_mean:.4f} (标准差: {dl_results['std_f1_macro']:.4f})\n")
                f.write(f"- **Subject 38性能**: {subject_38_f1:.4f}\n")
                f.write(f"- **性能差距**: {gap:.4f} ({gap/dl_results['std_f1_macro']:.1f} 个标准差)\n")
                
                if gap > 0.10:
                    f.write(f"- **严重性评估**: 显著的受试者个体差异问题\n")
                elif gap > 0.05:
                    f.write(f"- **严重性评估**: 中等程度的受试者差异\n")
                else:
                    f.write(f"- **严重性评估**: 轻微的受试者差异\n")
        
        # 主要建议
        if 'embedding_necessity_analysis' in self.results:
            final_assessment = self.results['embedding_necessity_analysis']['final_assessment']
            f.write(f"- **主要建议**: {final_assessment['recommendations']['overall_recommendation']}\n")
            f.write(f"- **实施优先级**: {final_assessment['recommendations']['priority']}\n")
            f.write(f"- **预期改善**: {final_assessment['recommendations']['estimated_improvement']}\n\n")

    def _write_data_overview(self, f):
        """写入数据概况"""
        f.write(f"- **总受试者数**: {self.data['n_subjects']}\n")
        f.write(f"- **总样本数**: {self.data['n_samples']:,}\n")
        f.write(f"- **特征维度**: {self.data['n_features']}\n")
        f.write(f"- **分类任务**: {self.data['n_classes']} 个脑区\n")
        f.write(f"- **年龄范围**: {self.subject_stats_df['age'].min():.1f} - {self.subject_stats_df['age'].max():.1f} 岁\n")
        f.write(f"- **平均每受试者样本数**: {self.subject_stats_df['n_samples'].mean():.0f}\n\n")

    def _write_loso_analysis(self, f):
        """写入LOSO分析结果"""
        if 'loso_analysis' not in self.results:
            f.write("LOSO分析未完成。\n\n")
            return
            
        f.write("### 深度学习LOSO性能分析\n\n")
        f.write("| 模型 | LOSO平均F1 | 标准差 | Subject 38 F1 | 性能差距 |\n")
        f.write("|------|------------|--------|---------------|----------|\n")
        
        for model_name, results in self.results['loso_analysis'].items():
            loso_mean = results['mean_f1_macro']
            loso_std = results['std_f1_macro']
            if results['subject_38_result']:
                subject_38_f1 = results['subject_38_result']['f1_macro']
                gap = loso_mean - subject_38_f1
                f.write(f"| {model_name} | {loso_mean:.4f} | {loso_std:.4f} | {subject_38_f1:.4f} | {gap:+.4f} |\n")
            else:
                f.write(f"| {model_name} | {loso_mean:.4f} | {loso_std:.4f} | N/A | N/A |\n")
        
        f.write("\n### LOSO分析解读\n\n")
        
        # ✅ 修复：使用DeepLearning结果
        dl_results = self.results['loso_analysis'].get('DeepLearning')  # ✅ 改成DeepLearning
        if dl_results:
            loso_mean = dl_results['mean_f1_macro']
            
            if loso_mean < 0.60:
                f.write("**结论**: 深度学习LOSO平均性能较低，表明受试者间差异是普遍存在的问题。这强烈支持使用Subject Embedding等个体化方法。\n\n")
            elif loso_mean > 0.70:
                f.write("**结论**: 深度学习LOSO平均性能良好，说明大多数受试者间的泛化是可行的。Subject 38的低性能可能是特殊情况。\n\n")
            else:
                f.write("**结论**: 深度学习LOSO平均性能中等，需要结合其他分析来确定最佳解决方案。\n\n")

    def _write_subject_38_analysis(self, f):
        """写入38号受试者分析结果"""
        if 'subject_38_analysis' not in self.results:
            f.write("Subject 38异常性分析未完成。\n\n")
            return
            
        anomaly_results = self.results['subject_38_analysis']
        
        f.write("### 异常性评估\n\n")
        anomaly_analysis = anomaly_results['anomaly_analysis']
        f.write(f"- **综合异常分数**: {anomaly_analysis['anomaly_score']}/{anomaly_analysis['max_possible_score']}\n")
        f.write(f"- **异常原因**: {', '.join(anomaly_analysis['anomaly_reasons'])}\n\n")
        
        f.write("### 详细分析结果\n\n")
        
        # 年龄分析
        age_analysis = anomaly_results['age_analysis']
        f.write(f"**年龄分析**:\n")
        f.write(f"- Subject 38年龄: {age_analysis['subject_38_age']:.1f} 岁\n")
        f.write(f"- 训练集平均年龄: {age_analysis['training_mean_age']:.1f} ± {age_analysis['training_std_age']:.1f} 岁\n")
        f.write(f"- 年龄差距: {age_analysis['age_difference']:+.1f} 岁\n")
        f.write(f"- 年龄Z分数: {age_analysis['age_z_score']:+.2f}\n\n")
        
        # 特征分析
        feature_analysis = anomaly_results['feature_analysis']
        f.write(f"**特征分析**:\n")
        f.write(f"- 异常特征比例 (2σ): {feature_analysis['outlier_percentage_2sigma']:.1f}%\n")
        f.write(f"- 严重异常特征 (3σ): {feature_analysis['outlier_percentage_3sigma']:.1f}%\n")
        f.write(f"- 最大|Z分数|: {feature_analysis['max_z_score']:.2f}\n\n")
        
        # 样本分析
        sample_analysis = anomaly_results['sample_analysis']
        f.write(f"**样本距离分析**:\n")
        f.write(f"- Subject 38平均距离: {sample_analysis['mean_distance_38']:.4f}\n")
        f.write(f"- 训练集平均距离: {sample_analysis['mean_distance_training']:.4f}\n")
        f.write(f"- 距离Z分数: {sample_analysis['distance_z_score']:+.2f}\n")
        f.write(f"- 异常样本比例: {sample_analysis['outlier_samples_percentage']:.1f}%\n\n")

    def _write_similarity_analysis(self, f):
        """写入受试者相似性分析结果"""
        if 'subject_similarity_analysis' not in self.results:
            f.write("受试者相似性分析未完成。\n\n")
            return
            
        similarity_results = self.results['subject_similarity_analysis']
        
        f.write("### 聚类分析结果\n\n")
        clustering = similarity_results['clustering']
        f.write(f"- **最优聚类数**: {clustering['best_n_clusters']}\n")
        f.write(f"- **聚类质量** (轮廓系数): {clustering['best_silhouette']:.3f}\n")
        
        if clustering['best_silhouette'] > 0.5:
            f.write("- **解读**: 受试者形成明显的聚类，建议考虑分层Subject Embedding\n\n")
        elif clustering['best_silhouette'] > 0.3:
            f.write("- **解读**: 受试者有一定的聚类趋势，可以考虑聚类导向的方法\n\n")
        else:
            f.write("- **解读**: 受试者聚类不明显，建议使用通用的Subject Embedding\n\n")
        
        # 38号相似性分析
        if 'subject_38_similarity' in similarity_results:
            f.write("### Subject 38 相似性分析\n\n")
            subject_38_sim = similarity_results['subject_38_similarity']
            f.write(f"- **最相似受试者**: {subject_38_sim['most_similar_subject']}\n")
            f.write(f"- **最小距离**: {subject_38_sim['min_distance']:.4f}\n")
            f.write(f"- **平均距离**: {subject_38_sim['mean_distance']:.4f}\n")
            f.write(f"- **距离排名**: {subject_38_sim['distance_rank']}/{len(similarity_results['subject_ids'])-1}\n\n")
        
        # 年龄-相似性关系
        age_corr = similarity_results['age_distance_correlation']
        f.write("### 年龄与相似性关系\n\n")
        f.write(f"- **年龄-距离相关性**: {age_corr:.4f}\n")
        
        if age_corr > 0.3:
            f.write("- **解读**: 年龄对受试者差异有显著影响，强烈建议年龄校正\n\n")
        elif age_corr > 0.1:
            f.write("- **解读**: 年龄对受试者差异有一定影响，建议考虑年龄因素\n\n")
        else:
            f.write("- **解读**: 年龄对受试者差异影响较小\n\n")
        

    def _write_age_effect_analysis(self, f):
        """写入年龄效应分析结果"""
        if 'age_effect_analysis' not in self.results:
            f.write("年龄效应分析未完成。\n\n")
            return
            
        age_results = self.results['age_effect_analysis']
        
        # 年龄分布基本信息
        age_dist = age_results['age_distribution']
        f.write("### 年龄分布基本信息\n\n")
        f.write(f"- **训练集年龄**: {age_dist['training_mean']:.1f} ± {age_dist['training_std']:.1f} 岁\n")
        f.write(f"- **训练集年龄范围**: [{age_dist['training_range'][0]:.1f}, {age_dist['training_range'][1]:.1f}] 岁\n")
        f.write(f"- **Subject 38年龄**: {age_dist['subject_38_age']:.1f} 岁\n")
        f.write(f"- **年龄差距**: {age_dist['age_gap']:+.1f} 岁\n")
        f.write(f"- **年龄Z分数**: {age_dist['age_z_score']:+.2f}\n\n")
        
        # 年龄异常性评估
        if abs(age_dist['age_z_score']) > 2:
            f.write("**年龄异常性评估**: 🚨 Subject 38年龄显著异常 (>2σ)\n\n")
        elif abs(age_dist['age_z_score']) > 1.5:
            f.write("**年龄异常性评估**: ⚠️ Subject 38年龄可能异常 (>1.5σ)\n\n")
        else:
            f.write("**年龄异常性评估**: ✅ Subject 38年龄在正常范围内\n\n")
        
        # 年龄校正建议
        f.write("### 年龄校正建议\n\n")
        correction_recs = age_results['correction_recommendations']
        f.write(f"- **校正优先级**: {correction_recs['correction_priority']}\n")
        f.write(f"- **推荐方法**: {', '.join(correction_recs['recommended_methods'])}\n")
        f.write(f"- **预期改善幅度**: {correction_recs['expected_improvement']:.3f}\n\n")
        
        # 年龄-性能关系分析
        if 'age_performance' in age_results:
            age_perf = age_results['age_performance']
            f.write("### 年龄-性能关系分析\n\n")
            f.write(f"- **年龄-性能相关性**: {age_perf['age_performance_correlation']:.4f}\n")
            f.write(f"- **基于年龄预测的Subject 38性能**: {age_perf['predicted_38_performance']:.4f}\n")
            
            if age_perf['actual_38_performance']:
                residual = age_perf['actual_38_performance'] - age_perf['predicted_38_performance']
                f.write(f"- **实际Subject 38性能**: {age_perf['actual_38_performance']:.4f}\n")
                f.write(f"- **年龄校正后残差**: {residual:+.4f}\n")
                
                if abs(residual) < 0.03:
                    f.write("- **年龄解释度**: 年龄基本解释了Subject 38的性能差异\n")
                    f.write("- **建议**: 主要使用年龄校正方法即可\n\n")
                elif abs(residual) < 0.05:
                    f.write("- **年龄解释度**: 年龄部分解释了Subject 38的性能差异\n")
                    f.write("- **建议**: 年龄校正结合轻量级Subject Embedding\n\n")
                else:
                    f.write("- **年龄解释度**: 年龄无法充分解释Subject 38的性能差异\n")
                    f.write("- **建议**: 需要完整的Subject Embedding解决方案\n\n")
            else:
                f.write("- **注意**: 无法获取Subject 38的实际性能数据进行比较\n\n")
        
        # 年龄相关性解读
        if 'age_performance' in age_results:
            age_corr = age_results['age_performance']['age_performance_correlation']
            f.write("### 年龄相关性解读\n\n")
            
            if abs(age_corr) > 0.3:
                f.write(f"**强相关性** (|r|={abs(age_corr):.3f}): 年龄对性能有显著影响，强烈建议年龄校正\n\n")
            elif abs(age_corr) > 0.1:
                f.write(f"**中等相关性** (|r|={abs(age_corr):.3f}): 年龄对性能有一定影响，建议考虑年龄因素\n\n")
            else:
                f.write(f"**弱相关性** (|r|={abs(age_corr):.3f}): 年龄对性能影响较小，可选择性校正\n\n")
        
        # 实施建议
        f.write("### 年龄校正实施建议\n\n")
        
        age_gap = abs(age_dist['age_gap'])
        if age_gap > 10:
            f.write("**实施方案**: 全面年龄校正\n")
            f.write("1. 建立年龄-性能回归模型\n")
            f.write("2. 对所有特征进行年龄去趋势处理\n")
            f.write("3. 在深度学习模型中添加年龄分支\n")
            f.write("4. 使用年龄分层的交叉验证\n\n")
        elif age_gap > 5:
            f.write("**实施方案**: 中等强度年龄校正\n")
            f.write("1. 在模型中添加年龄协变量\n")
            f.write("2. 进行年龄标准化处理\n")
            f.write("3. 监控年龄效应对结果的影响\n\n")
        else:
            f.write("**实施方案**: 轻量级年龄处理\n")
            f.write("1. 简单的年龄标准化\n")
            f.write("2. 可选的年龄协变量控制\n\n")



    def _write_embedding_recommendations(self, f):
        """写入Subject Embedding建议"""
        if 'embedding_necessity_analysis' not in self.results:
            f.write("Subject Embedding必要性分析未完成。\n\n")
            return
            
        embedding_results = self.results['embedding_necessity_analysis']
        final_assessment = embedding_results['final_assessment']
        
        f.write("### 必要性评估\n\n")
        f.write(f"- **综合必要性得分**: {final_assessment['necessity_score']}/{final_assessment['max_score']} ({final_assessment['necessity_percentage']:.1f}%)\n")
        f.write(f"- **总体建议**: {final_assessment['recommendations']['overall_recommendation']}\n")
        f.write(f"- **实施优先级**: {final_assessment['recommendations']['priority']}\n")
        f.write(f"- **预期改善**: {final_assessment['recommendations']['estimated_improvement']}\n\n")
        
        f.write("### 具体实施方法\n\n")
        for i, method in enumerate(final_assessment['recommendations']['specific_methods'], 1):
            f.write(f"{i}. {method}\n")
        
        f.write("\n### 架构建议\n\n")
        
        necessity_pct = final_assessment['necessity_percentage']
        if necessity_pct > 75:
            f.write("**推荐架构**: 完整的Subject Embedding系统\n")
            f.write("```python\n")
            f.write("# 推荐的双通道架构\n")
            f.write("subject_embedding_dim = 64\n")
            f.write("age_embedding_dim = 16\n")
            f.write("class SubjectAwareModel(nn.Module):\n")
            f.write("    def __init__(self):\n")
            f.write("        self.subject_embedding = nn.Embedding(n_subjects, subject_embedding_dim)\n")
            f.write("        self.age_projection = nn.Linear(1, age_embedding_dim)\n")
            f.write("        self.main_network = nn.Sequential(...)\n")
            f.write("        \n")
            f.write("    def forward(self, x, subject_id, age):\n")
            f.write("        subject_emb = self.subject_embedding(subject_id)\n")
            f.write("        age_emb = self.age_projection(age)\n")
            f.write("        combined = torch.cat([x, subject_emb, age_emb], dim=1)\n")
            f.write("        return self.main_network(combined)\n")
            f.write("```\n\n")
        elif necessity_pct > 50:
            f.write("**推荐架构**: 简化的Subject Embedding\n")
            f.write("```python\n")
            f.write("# 简化架构\n")
            f.write("subject_embedding_dim = 32\n")
            f.write("# 仅添加受试者嵌入层,结合年龄协变量\n")
            f.write("```\n\n")
        else:
            f.write("**推荐架构**: 轻量级年龄校正\n")
            f.write("```python\n")
            f.write("# 年龄校正方法\n")
            f.write("age_corrected_features = features - age_effect_model(age)\n")
            f.write("```\n\n")
        
    def _write_implementation_roadmap(self, f):
        """写入实施路线图"""
        f.write("基于分析结果,建议按以下优先级实施改进措施:\n\n")
        
        # 基于分析结果生成优先级
        priorities = []
        
        # 年龄校正优先级
        if 'age_effect_analysis' in self.results:
            age_gap = abs(self.results['age_effect_analysis']['age_distribution']['age_gap'])
            if age_gap > 5:
                priorities.append(("年龄校正", "高", f"年龄差距{age_gap:.1f}岁"))
        
        # Subject Embedding优先级
        if 'embedding_necessity_analysis' in self.results:
            necessity_pct = self.results['embedding_necessity_analysis']['final_assessment']['necessity_percentage']
            if necessity_pct > 50:
                priority_level = "高" if necessity_pct > 75 else "中"
                priorities.append(("Subject Embedding", priority_level, f"必要性{necessity_pct:.1f}%"))
        
        # 数据质量检查优先级
        if 'subject_38_analysis' in self.results:
            anomaly_score = self.results['subject_38_analysis']['anomaly_analysis']['anomaly_score']
            if anomaly_score > 5:
                priorities.append(("数据质量检查", "高", f"异常分数{anomaly_score}"))
        
        # LOSO性能优先级
        if 'loso_analysis' in self.results:
            dl_results = self.results['loso_analysis'].get('DeepLearning')  # ✅ 改成DeepLearning
            if dl_results and dl_results['mean_f1_macro'] < 0.6:
                priorities.append(("域适应方法", "中", "LOSO性能较低"))
        
        # 排序并输出
        priority_order = {"高": 1, "中": 2, "低": 3}
        priorities.sort(key=lambda x: priority_order.get(x[1], 4))
        
        for i, (method, priority, reason) in enumerate(priorities, 1):
            f.write(f"### {i}. {method} (优先级: {priority})\n\n")
            f.write(f"**原因**: {reason}\n\n")
            
            if method == "年龄校正":
                f.write("**实施步骤**:\n")
                f.write("1. 实施年龄回归校正\n")
                f.write("2. 在模型中添加年龄协变量\n")
                f.write("3. 评估年龄校正效果\n\n")
                f.write("**预期时间**: 1-2周\n\n")
                
            elif method == "Subject Embedding":
                f.write("**实施步骤**:\n")
                f.write("1. 设计Subject Embedding架构\n")
                f.write("2. 修改训练流程支持受试者ID\n")
                f.write("3. 训练和评估新模型\n")
                f.write("4. 超参数调优\n\n")
                f.write("**预期时间**: 2-3周\n\n")
                
            elif method == "数据质量检查":
                f.write("**实施步骤**:\n")
                f.write("1. 检查Subject 38的数据采集参数\n")
                f.write("2. 对比扫描质量指标\n")
                f.write("3. 考虑数据预处理差异\n")
                f.write("4. 必要时重新采集或排除\n\n")
                f.write("**预期时间**: 1周\n\n")
                
            elif method == "域适应方法":
                f.write("**实施步骤**:\n")
                f.write("1. 实施域对抗训练\n")
                f.write("2. 尝试特征对齐方法\n")
                f.write("3. 考虑元学习方法\n\n")
                f.write("**预期时间**: 3-4周\n\n")
        
        f.write("### 评估指标\n\n")
        f.write("在实施过程中,建议监控以下指标:\n\n")
        f.write("1. **LOSO交叉验证性能**: 目标提升到0.65+\n")
        f.write("2. **Subject 38性能**: 目标缩小与LOSO平均值的差距到0.05以内\n")
        f.write("3. **年龄校正效果**: 年龄-性能相关性降低到0.1以下\n")
        f.write("4. **模型复杂度**: 保持训练时间在可接受范围内\n\n")
        
        f.write("### 风险评估\n\n")
        f.write("**主要风险**:\n")
        f.write("- Subject Embedding可能导致过拟合\n")
        f.write("- 年龄校正可能移除有用信息\n")
        f.write("- 实施复杂度增加\n\n")
        
        f.write("**缓解措施**:\n")
        f.write("- 使用适当的正则化\n")
        f.write("- 保守的年龄校正策略\n")
        f.write("- 分步骤实施和评估\n\n")

    
    def _prepare_results_for_json(self):
        """准备结果用于JSON序列化"""
        json_results = {}
        
        for key, value in self.results.items():
            json_results[key] = self._convert_to_serializable(value)
            
        # 添加受试者统计信息
        json_results['subject_statistics'] = self.subject_stats_df.to_dict('records')
        
        return json_results
    
    def _convert_to_serializable(self, obj):
        """递归转换对象为JSON可序列化格式"""
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, dict):
            return {k: self._convert_to_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [self._convert_to_serializable(item) for item in obj]
        elif isinstance(obj, pd.DataFrame):
            return obj.to_dict('records')
        else:
            return obj
    
    def run_complete_analysis(self):
        """
        运行完整的受试者个体差异分析流程
        
        Returns:
            dict: 包含所有分析结果的字典
        """
        
        print("🚀 开始完整的受试者个体差异分析")
        print("="*80)
        start_time = time.time()
        
        try:
            # Phase 1: 数据加载
            print("\n🔄 Phase 1: 数据加载...")
            self.load_data_with_subjects_and_age()
            
            # Phase 2: LOSO分析
            print("\n🔄 Phase 2: LOSO分析...")
            self.leave_one_subject_out_analysis()
            
            # Phase 3: 38号受试者异常性分析
            print("\n🔄 Phase 3: Subject 38异常性分析...")
            self.analyze_subject_38_anomaly()
            
            # Phase 4: 受试者相似性分析
            print("\n🔄 Phase 4: 受试者相似性分析...")
            self.analyze_subject_similarity_and_clustering()
            
            # Phase 5: 年龄效应分析
            print("\n🔄 Phase 5: 年龄效应分析...")
            self.analyze_age_effects()
            
            # Phase 6: Subject Embedding必要性评估
            print("\n🔄 Phase 6: Subject Embedding必要性评估...")
            self.evaluate_embedding_necessity()
            
            # Phase 7: 生成可视化
            print("\n🔄 Phase 7: 生成可视化...")
            self.generate_comprehensive_visualizations()
            
            # Phase 8: 生成报告
            print("\n🔄 Phase 8: 生成分析报告...")
            report_path, results_path = self.generate_comprehensive_report()
            
            # 计算运行时间
            end_time = time.time()
            runtime = end_time - start_time
            
            print(f"\n🎉 完整分析已完成!")
            print(f"📁 结果保存在: {self.save_path}")
            print(f"⏱️ 总运行时间: {runtime:.2f} 秒")
            print(f"📄 报告路径: {report_path}")
            
            # 生成最终总结
            self._print_final_summary()
            
            # 返回完整结果
            return {
                'analysis_results': self.results,
                'subject_stats': self.subject_stats_df,
                'runtime': runtime,
                'save_path': self.save_path,
                'report_path': report_path,
                'results_path': results_path
            }
            
        except Exception as e:
            print(f"❌ 分析过程中出现错误: {e}")
            import traceback
            traceback.print_exc()
            return None
    
    def _print_final_summary(self):
        """打印最终总结"""
        print("\n" + "="*80)
        print("📋 SUBJECT VARIABILITY ANALYSIS - FINAL SUMMARY")
        print("="*80)
        
        # LOSO结果总结
        if 'loso_analysis' in self.results and 'DeepLearning' in self.results['loso_analysis']:  # ✅ 改成DeepLearning
            dl_results = self.results['loso_analysis']['DeepLearning']  # ✅ 改成DeepLearning
            print(f"\n🎯 LOSO Analysis Results:")
            print(f"  - Mean Performance: {dl_results['mean_f1_macro']:.4f} ± {dl_results['std_f1_macro']:.4f}")
            print(f"  - Performance Range: [{dl_results['min_f1_macro']:.4f}, {dl_results['max_f1_macro']:.4f}]")
            
            if dl_results['subject_38_result']:
                subject_38_f1 = dl_results['subject_38_result']['f1_macro']
                gap = dl_results['mean_f1_macro'] - subject_38_f1
                print(f"  - Subject 38 Performance: {subject_38_f1:.4f}")
                print(f"  - Performance Gap: {gap:+.4f}")
        
        # 异常性分析总结
        if 'subject_38_analysis' in self.results:
            anomaly_analysis = self.results['subject_38_analysis']['anomaly_analysis']
            print(f"\n🔍 Subject 38 Anomaly Analysis:")
            print(f"  - Anomaly Score: {anomaly_analysis['anomaly_score']}/{anomaly_analysis['max_possible_score']}")
            if anomaly_analysis['anomaly_reasons']:
                print(f"  - Main Issues: {', '.join(anomaly_analysis['anomaly_reasons'][:3])}")
        
        # 年龄效应总结
        if 'age_effect_analysis' in self.results:
            age_dist = self.results['age_effect_analysis']['age_distribution']
            print(f"\n📊 Age Effect Analysis:")
            print(f"  - Age Gap: {age_dist['age_gap']:+.1f} years")
            print(f"  - Age Z-Score: {age_dist['age_z_score']:+.2f}")
        
        # 最终建议
        if 'embedding_necessity_analysis' in self.results:
            final_assessment = self.results['embedding_necessity_analysis']['final_assessment']
            print(f"\n💡 Final Recommendations:")
            print(f"  - Overall Recommendation: {final_assessment['recommendations']['overall_recommendation']}")
            print(f"  - Priority: {final_assessment['recommendations']['priority']}")
            print(f"  - Expected Improvement: {final_assessment['recommendations']['estimated_improvement']}")
            print(f"  - Necessity Score: {final_assessment['necessity_percentage']:.1f}%")
        
        print(f"\n📁 All results saved in: {self.save_path}")
        print("="*80)


# ============================================================================
# 🚀 主执行函数和使用示例
# ============================================================================


def run_subject_variability_analysis(data_path, save_path='./subject_variability_analysis/'):
    """
    运行完整的受试者个体差异分析
    
    Args:
        data_path: TRAIN38.mat文件路径
        save_path: 结果保存路径
    
    Returns:
        analyzer: 分析器对象，包含所有结果
        results: 分析结果字典
    """
    
    print("🧠 开始受试者个体差异分析...")
    print("="*80)
    
    # 初始化分析器
    analyzer = SubjectVariabilityAnalyzer(data_path=data_path, save_path=save_path)
    
    # 运行完整分析
    results = analyzer.run_complete_analysis()
    
    if results is None:
        print("❌ 分析失败")
        return None, None
    
    print("\n🎉 Subject Variability Analysis 完成!")
    print("="*80)
    print(f"📊 分析总结:")
    
    # 提取关键结果进行总结
    if 'loso_analysis' in analyzer.results and 'DeepLearning' in analyzer.results['loso_analysis']:
        dl_results = analyzer.results['loso_analysis']['DeepLearning']
        loso_mean = dl_results['mean_f1_macro']
        
        if dl_results['subject_38_result']:
            subject_38_f1 = dl_results['subject_38_result']['f1_macro']
            gap = loso_mean - subject_38_f1
            
            print(f"  🎯 LOSO平均F1: {loso_mean:.4f}")
            print(f"  🎯 Subject 38 F1: {subject_38_f1:.4f}")
            print(f"  🎯 性能差距: {gap:.4f}")
            
            if gap > 0.10:
                print(f"  💡 结论: 显著的受试者个体差异，强烈建议Subject Embedding")
            elif gap > 0.05:
                print(f"  💡 结论: 中等程度受试者差异，建议Subject Embedding")
            else:
                print(f"  💡 结论: 轻微受试者差异，可选择性使用Subject Embedding")
    
    if 'embedding_necessity_analysis' in analyzer.results:
        final_assessment = analyzer.results['embedding_necessity_analysis']['final_assessment']
        print(f"  🎯 最终建议: {final_assessment['recommendations']['overall_recommendation']}")
    
    return analyzer, results

# ============================================================================
# 🔧 便捷函数：快速诊断特定问题
# ============================================================================

def quick_loso_analysis(data_path):
    """
    快速LOSO分析，只运行核心的Leave-One-Subject-Out评估
    """
    analyzer = SubjectVariabilityAnalyzer(data_path=data_path, save_path='./temp_loso/')
    analyzer.load_data_with_subjects_and_age()
    loso_results = analyzer.leave_one_subject_out_analysis()
    
    return loso_results

def quick_subject_38_diagnosis(data_path):
    """
    快速Subject 38异常性诊断
    """
    analyzer = SubjectVariabilityAnalyzer(data_path=data_path, save_path='./temp_38_diagnosis/')
    analyzer.load_data_with_subjects_and_age()
    anomaly_results = analyzer.analyze_subject_38_anomaly()
    
    return anomaly_results

def compare_multiple_models_loso(data_path, custom_models=None):
    """
    对比多个模型的LOSO性能
    """
    analyzer = SubjectVariabilityAnalyzer(data_path=data_path, save_path='./temp_multi_model/')

    
    analyzer.load_data_with_subjects_and_age()
    loso_results = analyzer.leave_one_subject_out_analysis()
    
    # 生成对比图表
    analyzer.generate_comprehensive_visualizations()
    
    return loso_results

# ============================================================================
# 💡 使用示例
# ============================================================================

if __name__ == "__main__":
    # 设置数据路径
    DATA_PATH = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat'
    SAVE_PATH = './subject_variability_analysis_results/'
    
    # 运行完整分析
    print("🚀 开始受试者个体差异分析...")
    analyzer, results = run_subject_variability_analysis(
        data_path=DATA_PATH,
        save_path=SAVE_PATH
    )
    
    if analyzer and results:
        print(f"\n✅ 分析完成！所有结果保存在: {SAVE_PATH}")
        print(f"📊 查看详细报告: {results['report_path']}")
        
        # 可以进一步访问具体结果
        if 'loso_analysis' in analyzer.results:
            print(f"\n🔍 快速查看LOSO结果:")
            for model_name, model_results in analyzer.results['loso_analysis'].items():
                print(f"  {model_name}: {model_results['mean_f1_macro']:.4f} ± {model_results['std_f1_macro']:.4f}")
        
        # 访问Subject Embedding建议
        if 'embedding_necessity_analysis' in analyzer.results:
            embedding_rec = analyzer.results['embedding_necessity_analysis']['final_assessment']
            print(f"\n💡 Subject Embedding建议:")
            print(f"  必要性: {embedding_rec['necessity_percentage']:.1f}%")
            print(f"  建议: {embedding_rec['recommendations']['overall_recommendation']}")
            print(f"  预期改善: {embedding_rec['recommendations']['estimated_improvement']}")
    else:
        print("❌ 分析失败，请检查数据路径和参数设置")